# Prediksi Tinggi Muka Air Sungai (v2)

Pipeline end-to-end untuk audit data, feature engineering lingkungan dan HydroRIVERS, rolling-origin validation, ensemble multi-model, dan pembuatan submission.

**Guard training:** notebook memakai `MODE = "audit"` secara default. Menjalankan seluruh cell tidak akan melakukan fitting sampai mode diubah secara eksplisit. Desain dan hasil audit lengkap ada di `architecture.md` dan `logs.md`.

**Perubahan v2 (tindak lanjut review senior, 15 Juli 2026):**

1. **Validasi ensemble, bukan satu seed.** `run_validation` sekarang melatih dan merata-ratakan seluruh anggota ensemble di setiap fold, sama seperti submission final — bukan cuma seed pertama. Angka RMSE yang dilaporkan sekarang benar-benar merepresentasikan apa yang dikirim ke leaderboard.
2. **Ensemble terdiversifikasi.** Bukan 3 CatBoost dengan hyperparameter identik (cuma beda seed), sekarang 3 varian CatBoost (depth berbeda) + 2 varian LightGBM (jumlah leaves/feature-subsampling berbeda). Semua anggota tetap memprediksi target robust-normalized yang sama sehingga bisa dirata-rata langsung di skala `z`.
3. **Ablasi spike-downweighting dan feature pruning.** Mode baru `MODE = "ablation"` menjalankan on/off test murah (1 model cepat, 2 fold terakhir) untuk memvalidasi apakah spike-downweighting dan feature pruning (pakai `feature_importance.csv` yang sudah dihitung) benar-benar menaikkan metrik, bukan cuma asumsi desain.
4. **Hyperparameter sweep ringan + model quantile opsional.** Mode baru `MODE = "tune"` menjalankan sweep depth/learning_rate/l2 CatBoost pada satu fold representatif. `ENABLE_QUANTILE_MODEL = True` melatih satu CatBoost `MultiQuantile` tambahan untuk interval prediksi (q10/q50/q90) di samping titik prediksi utama.

Urutan eksekusi yang disarankan: `audit` -> `tune` -> `validate` (untuk menghasilkan `feature_importance.csv`) -> `ablation` -> `fit`/`all` dengan konfigurasi final.


## 1. Import, path, dan konfigurasi model


In [57]:
"""Training and submission pipeline for river water-level (TMA) forecasting.

The pipeline is deliberately exogenous and non-recursive: every feature used for a
future prediction is available in ``data_lingkungan.csv`` or is static/calendar
information.  This avoids compounding autoregressive error over the 242-day test
horizon.

v2 additions (senior review follow-up, 2026-07-15):
- Validation now scores the FULL ensemble per fold instead of a single seed, so the
  reported metric matches what actually goes into the submission.
- The ensemble is diversified across CatBoost (varying depth/lr/l2) and LightGBM
  (varying leaves/feature-subsampling) members instead of 3 identical CatBoost seeds.
- Spike-downweighting and feature-pruning are togglable so they can be ablated
  instead of trusted on intuition alone.
- Optional lightweight hyperparameter sweep and optional CatBoost quantile model.

Training is never started implicitly. The execution cell defaults to audit mode.
"""

from __future__ import annotations

import importlib.metadata
import json
import logging
import math
import platform
import re
import sys
import traceback
import zipfile
from dataclasses import asdict, dataclass, replace
from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable, Sequence

import geopandas as gpd
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error


LOGGER = logging.getLogger("tma_forecasting")
KEYS = ["datetime", "nama_pos"]
ID_PATTERN = re.compile(
    r"^(?P<datetime>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) - (?P<nama_pos>.+)$"
)


@dataclass(frozen=True)
class Paths:
    root: Path = Path("datasets")
    artifacts: Path = Path("artifacts")

    @property
    def train(self) -> Path:
        return self.root / "train.csv"

    @property
    def test(self) -> Path:
        return self.root / "test.csv"

    @property
    def sample_submission(self) -> Path:
        return self.root / "sample_submission.csv"

    @property
    def coordinates(self) -> Path:
        return self.root / "data_pendukung" / "koordinat_pos.csv"

    @property
    def environment(self) -> Path:
        return self.root / "data_pendukung" / "data_lingkungan.csv"

    @property
    def rivers(self) -> Path:
        return (
            self.root
            / "data_pendukung"
            / "HydroRIVERS_v10_au_shp"
            / "HydroRIVERS_v10_au.shp"
        )


@dataclass(frozen=True)
class EnsembleMemberConfig:
    """One ensemble member.

    Members are intentionally heterogeneous (family / depth / leaves / feature
    subsampling) so the ensemble reduces *structural* bias, not just seed variance.
    Every member still predicts the same robust-normalized per-station target, so
    predictions stay directly averageable on the z-scale regardless of family.
    """

    name: str
    family: str  # "catboost" or "lightgbm"
    seed: int
    # CatBoost-specific
    depth: int = 8
    learning_rate: float = 0.035
    l2_leaf_reg: float = 10.0
    # LightGBM-specific
    num_leaves: int = 63
    feature_fraction: float = 0.9
    bagging_fraction: float = 0.85
    min_child_samples: int = 25


DEFAULT_ENSEMBLE: tuple[EnsembleMemberConfig, ...] = (
    EnsembleMemberConfig("cb_shallow", "catboost", seed=17, depth=6, learning_rate=0.045, l2_leaf_reg=8.0),
    EnsembleMemberConfig("cb_base", "catboost", seed=41, depth=8, learning_rate=0.02, l2_leaf_reg=10.0),
    EnsembleMemberConfig("cb_deep", "catboost", seed=83, depth=8, learning_rate=0.045, l2_leaf_reg=6.0),
    EnsembleMemberConfig("lgbm_a", "lightgbm", seed=7, num_leaves=63, learning_rate=0.03,
                          feature_fraction=0.85, bagging_fraction=0.8),
    EnsembleMemberConfig("lgbm_b", "lightgbm", seed=29, num_leaves=127, learning_rate=0.02,
                          feature_fraction=0.7, bagging_fraction=0.8),
)


@dataclass(frozen=True)
class ModelConfig:
    ensemble: tuple[EnsembleMemberConfig, ...] = DEFAULT_ENSEMBLE
    default_iterations: int = 1800
    huber_delta: float = 1.5  # CatBoost Huber delta
    lightgbm_huber_alpha: float = 1.5
    early_stopping_rounds: int = 200
    early_stopping_metric: str = "current"  # current | rmse
    verbose_every: int = 100
    upstream_decay_km: float = 75.0
    snapshot_interval_seconds: int = 300
    # --- Ablation / optimization toggles (senior review points 1-4) ---
    use_spike_weights: bool = True
    prune_features: bool = False
    feature_importance_source: str | None = None  # path to a prior feature_importance.csv
    prune_keep_fraction: float = 0.85
    enable_quantile_model: bool = False
    quantile_levels: tuple[float, ...] = (0.1, 0.5, 0.9)
    # --- Leakage-safe target formulation ---
    target_mode: str = "raw_robust"  # raw_robust | harmonic_residual
    # --- Per-station harmonic baseline feature (Wonogiri Dam follow-up) ---
    enable_harmonic_feature: bool = True
    harmonic_ridge_alpha: float = 5.0
    # --- Recency/persistence anchor feature (station-aware stacking follow-up) ---
    enable_anchor_feature: bool = True
    anchor_recent_window_days: float = 30.0
    anchor_tau_days: float = 365.0
    # --- GPU acceleration ---
    use_gpu: bool = True
    gpu_device_id: str = "0"
    # Leakage-safe learned ensemble weighting.
    enable_learned_ensemble_weights: bool = True
    blend_ridge_strength: float = 0.05
    # These folds exactly match the Sep-to-May competition forecast season.
    blend_fit_folds: tuple[str, ...] = ("sep_2023", "sep_2024")


FOLDS = (
    ("sep_2023", "2023-09-18 18:00:00", "2023-09-19 06:00:00", "2024-05-18 18:00:00"),
    ("may_2024", "2024-05-18 18:00:00", "2024-05-19 06:00:00", "2025-01-18 18:00:00"),
    ("sep_2024", "2024-09-18 18:00:00", "2024-09-19 06:00:00", "2025-05-18 18:00:00"),
    ("jan_2025", "2025-01-18 18:00:00", "2025-01-19 06:00:00", "2025-09-18 18:00:00"),
)


RIVER_NUMERIC = [
    "LENGTH_KM",
    "DIST_DN_KM",
    "DIST_UP_KM",
    "CATCH_SKM",
    "UPLAND_SKM",
    "DIS_AV_CMS",
    "ORD_STRA",
    "ORD_CLAS",
    "ORD_FLOW",
]

RAW_STATE_COLUMNS = [
    "soil_moisture_0_7cm",
    "soil_moisture_7_28cm",
    "soil_moisture_28_100cm",
    "soil_moisture_100_255cm",
    "surface_pressure_hpa",
    "pressure_msl_hpa",
    "rmm1",
    "rmm2",
    "mjo_phase",
    "mjo_amplitude",
    "mjo_active",
    "nino_34",
]

CATEGORICAL_COLUMNS = [
    "nama_pos",
    "landcover_name",
    "hour_cat",
    "month_cat",
    "mjo_phase_cat",
    "main_riv_cat",
    "hybas_l12_cat",
    "landcover_class_cat",
    "endorheic_cat",
]

NON_FEATURE_COLUMNS = {
    "datetime",
    "id",
    "tma_mdpl",
    "_row_order",
    "mjo_phase",
    "landcover_class",
    "HYRIV_ID",
    "NEXT_DOWN",
    "MAIN_RIV",
    "HYBAS_L12",
    "ENDORHEIC",
    "wind_direction_deg",
}


## 2. Parsing ID, pemuatan data, dan audit schema

Missing baris target tidak dibuat menjadi pseudo-label. Semua key, station set, jam, dan urutan sample submission diperiksa sebelum feature engineering.


In [58]:
def parse_submission_ids(frame: pd.DataFrame) -> pd.DataFrame:
    """Parse IDs without splitting station names that themselves contain ' - '."""
    parsed = frame["id"].str.extract(ID_PATTERN)
    if parsed.isna().any(axis=None):
        bad = frame.loc[parsed.isna().any(axis=1), "id"].head(3).tolist()
        raise ValueError(f"Invalid submission IDs; examples: {bad}")
    result = frame.copy()
    result["datetime"] = pd.to_datetime(parsed["datetime"], errors="raise")
    result["nama_pos"] = parsed["nama_pos"]
    result["_row_order"] = np.arange(len(result), dtype=np.int64)
    return result


def load_inputs(paths: Paths) -> tuple[pd.DataFrame, ...]:
    train = pd.read_csv(paths.train, parse_dates=["datetime"])
    test_raw = pd.read_csv(paths.test)
    sample = pd.read_csv(paths.sample_submission)
    coordinates = pd.read_csv(paths.coordinates)
    environment = pd.read_csv(paths.environment, parse_dates=["datetime"])
    test = parse_submission_ids(test_raw)
    return train, test, sample, coordinates, environment


def audit_inputs(
    train: pd.DataFrame,
    test: pd.DataFrame,
    sample: pd.DataFrame,
    coordinates: pd.DataFrame,
    environment: pd.DataFrame,
) -> dict:
    """Fail early on silent schema, key, time, or station-set changes."""
    expected_train = {"datetime", "nama_pos", "tma_mdpl"}
    if set(train.columns) != expected_train:
        raise ValueError(f"Unexpected train schema: {train.columns.tolist()}")
    if train.duplicated(KEYS).any() or environment.duplicated(KEYS).any():
        raise ValueError("Duplicate (datetime, nama_pos) keys detected")
    if train["tma_mdpl"].isna().any():
        raise ValueError("Target contains explicit NaN values")
    if test["id"].duplicated().any():
        raise ValueError("Duplicate test IDs detected")
    if not test["id"].reset_index(drop=True).equals(sample["id"].reset_index(drop=True)):
        raise ValueError("test.csv and sample_submission.csv IDs/order differ")

    train_stations = set(train["nama_pos"])
    station_sets = {
        "test": set(test["nama_pos"]),
        "coordinates": set(coordinates["nama_pos"]),
        "environment": set(environment["nama_pos"]),
    }
    for label, stations in station_sets.items():
        if stations != train_stations:
            raise ValueError(
                f"Station mismatch for {label}: missing={sorted(train_stations-stations)}, "
                f"extra={sorted(stations-train_stations)}"
            )

    expected_hours = {6, 12, 18}
    if set(train["datetime"].dt.hour) - expected_hours:
        raise ValueError("Train contains target hours outside 06, 12, 18")
    if set(test["datetime"].dt.hour) != expected_hours:
        raise ValueError("Test hours are not exactly 06, 12, 18")

    env_counts = environment.groupby("nama_pos", observed=True).size()
    expected_env_rows = len(
        pd.date_range(environment["datetime"].min(), environment["datetime"].max(), freq="h")
    )
    if not (env_counts == expected_env_rows).all():
        raise ValueError("Environment is not a complete hourly grid for every station")

    return {
        "train_rows": int(len(train)),
        "test_rows": int(len(test)),
        "environment_rows": int(len(environment)),
        "stations": int(len(train_stations)),
        "train_start": str(train["datetime"].min()),
        "train_end": str(train["datetime"].max()),
        "test_start": str(test["datetime"].min()),
        "test_end": str(test["datetime"].max()),
        "environment_missing_cells": int(environment.isna().sum().sum()),
        "solar_minus_999_cells": int((environment["solar_radiation_mj_m2"] <= -900).sum()),
    }


## 3. HydroRIVERS dan graf upstream

Koordinat pos dicocokkan dalam UTM 49S. Traversal `NEXT_DOWN` membentuk bobot lingkungan upstream yang menurun terhadap jarak hidraulik.


In [59]:
def _read_local_rivers(path: Path, coordinates: pd.DataFrame) -> gpd.GeoDataFrame:
    pad = 0.75
    bbox = (
        coordinates["longitude"].min() - pad,
        coordinates["latitude"].min() - pad,
        coordinates["longitude"].max() + pad,
        coordinates["latitude"].max() + pad,
    )
    rivers = gpd.read_file(path, bbox=bbox)
    required = {
        "HYRIV_ID",
        "NEXT_DOWN",
        "MAIN_RIV",
        "HYBAS_L12",
        "ENDORHEIC",
        *RIVER_NUMERIC,
        "geometry",
    }
    missing = required - set(rivers.columns)
    if missing:
        raise ValueError(f"HydroRIVERS is missing columns: {sorted(missing)}")
    if rivers.crs is None:
        raise ValueError("HydroRIVERS has no CRS")
    return rivers


def match_stations_to_rivers(
    coordinates: pd.DataFrame, rivers: gpd.GeoDataFrame
) -> pd.DataFrame:
    points = gpd.GeoDataFrame(
        coordinates.copy(),
        geometry=gpd.points_from_xy(coordinates["longitude"], coordinates["latitude"]),
        crs="EPSG:4326",
    )
    # All supplied gauges are inside UTM zone 49S (108E-114E).
    points_utm = points.to_crs("EPSG:32749")
    rivers_utm = rivers.to_crs("EPSG:32749")
    nearest = gpd.sjoin_nearest(
        points_utm,
        rivers_utm,
        how="left",
        distance_col="river_distance_m",
    )
    nearest = (
        nearest.sort_values("river_distance_m")
        .loc[lambda x: ~x.index.duplicated(keep="first")]
        .sort_index()
    )
    if nearest["HYRIV_ID"].isna().any():
        raise ValueError("At least one station could not be matched to HydroRIVERS")

    keep = [
        "nama_pos",
        "latitude",
        "longitude",
        "river_distance_m",
        "HYRIV_ID",
        "NEXT_DOWN",
        "MAIN_RIV",
        "HYBAS_L12",
        "ENDORHEIC",
        *RIVER_NUMERIC,
    ]
    static = pd.DataFrame(nearest[keep]).reset_index(drop=True)
    static["river_match_confidence"] = np.exp(-static["river_distance_m"] / 1000.0)
    for col in ["CATCH_SKM", "UPLAND_SKM", "DIS_AV_CMS", "LENGTH_KM"]:
        static[f"log1p_{col.lower()}"] = np.log1p(static[col].clip(lower=0))
    static["main_riv_cat"] = static["MAIN_RIV"].astype("Int64").astype(str)
    static["hybas_l12_cat"] = static["HYBAS_L12"].astype("Int64").astype(str)
    static["endorheic_cat"] = static["ENDORHEIC"].astype("Int64").astype(str)
    return static


def build_upstream_weights(
    static: pd.DataFrame,
    rivers: gpd.GeoDataFrame,
    decay_km: float,
) -> pd.DataFrame:
    """Create target-by-source weights from exact NEXT_DOWN reach traversal."""
    stations = static["nama_pos"].tolist()
    reach_by_station = dict(zip(static["nama_pos"], static["HYRIV_ID"].astype(int)))
    dist_by_station = dict(zip(static["nama_pos"], static["DIST_DN_KM"].astype(float)))
    station_by_reach = {reach: station for station, reach in reach_by_station.items()}
    next_down = dict(zip(rivers["HYRIV_ID"].astype(int), rivers["NEXT_DOWN"].astype(int)))

    weights = pd.DataFrame(0.0, index=stations, columns=stations)
    for source in stations:
        weights.loc[source, source] = 1.0
        current = reach_by_station[source]
        seen = {current}
        for _ in range(len(next_down) + 1):
            current = next_down.get(current, 0)
            if current == 0 or current in seen:
                break
            seen.add(current)
            target = station_by_reach.get(current)
            if target is not None:
                distance = max(0.0, dist_by_station[source] - dist_by_station[target])
                weights.loc[target, source] = max(
                    weights.loc[target, source], math.exp(-distance / decay_km)
                )
    return weights.div(weights.sum(axis=1), axis=0)


def _add_upstream_environment(
    environment: pd.DataFrame,
    weights: pd.DataFrame,
    source_columns: Iterable[str],
) -> pd.DataFrame:
    result = environment
    station_order = weights.columns.tolist()
    for column in source_columns:
        pivot = result.pivot(index="datetime", columns="nama_pos", values=column).reindex(
            columns=station_order
        )
        upstream = pd.DataFrame(
            pivot.to_numpy(dtype=float) @ weights.to_numpy(dtype=float).T,
            index=pivot.index,
            columns=weights.index,
        )
        long = (
            upstream.rename_axis(columns="nama_pos")
            .stack(future_stack=True)
            .rename(f"upstream_{column}")
            .reset_index()
        )
        result = result.merge(long, on=KEYS, how="left", validate="one_to_one")
    return result


## 4. Sanitasi lingkungan dan fitur antecedent

Menangani sentinel radiasi `-999`, missing tail, arah angin siklik, hujan/soil upstream, rolling window, dan antecedent precipitation index secara kausal.


In [60]:
def prepare_environment(
    environment: pd.DataFrame, upstream_weights: pd.DataFrame
) -> pd.DataFrame:
    env = environment.sort_values(KEYS).copy()

    # These columns are byte-identical in the supplied data; retain one copy only.
    if not env["rainfall_mm"].equals(env["rainfall_openmeteo_mm"]):
        raise ValueError("Rainfall duplicate columns have unexpectedly diverged")
    env = env.drop(columns="rainfall_openmeteo_mm")

    # -999 is an undocumented sentinel, not physically possible solar radiation.
    env["solar_radiation_missing"] = (
        env["solar_radiation_mj_m2"].isna()
        | (env["solar_radiation_mj_m2"] <= -900)
    ).astype("int8")
    env.loc[env["solar_radiation_mj_m2"] <= -900, "solar_radiation_mj_m2"] = np.nan

    # Missing indicators are retained even after causal forward filling.
    for col in RAW_STATE_COLUMNS:
        env[f"{col}_missing"] = env[col].isna().astype("int8")

    by_station = env.groupby("nama_pos", sort=False, observed=True)
    short_state = [c for c in RAW_STATE_COLUMNS if c != "nino_34"]
    env[short_state] = by_station[short_state].ffill(limit=24)
    # Nino 3.4 is a slowly varying monthly index; a 31-day causal hold is appropriate.
    env[["nino_34"]] = by_station[["nino_34"]].ffill(limit=31 * 24)

    radians = np.deg2rad(env["wind_direction_deg"])
    env["wind_direction_sin"] = np.sin(radians)
    env["wind_direction_cos"] = np.cos(radians)
    env["dewpoint_depression_c"] = env["temperature_c"] - env["dew_point_c"]

    env = _add_upstream_environment(
        env,
        upstream_weights,
        ["rainfall_mm", "soil_moisture_0_7cm"],
    )
    env = env.sort_values(KEYS).reset_index(drop=True)
    grouped = env.groupby("nama_pos", sort=False, observed=True)

    rolling_sum_specs = {
        "rainfall_mm": (6, 24, 72, 168, 336),
        "upstream_rainfall_mm": (6, 24, 72, 168, 336),
    }
    for col, windows in rolling_sum_specs.items():
        for hours in windows:
            env[f"{col}_sum_{hours}h"] = grouped[col].transform(
                lambda s, w=hours: s.rolling(w, min_periods=max(1, w // 4)).sum()
            )
        env[f"{col}_max_24h"] = grouped[col].transform(
            lambda s: s.rolling(24, min_periods=6).max()
        )
        for half_life in (12, 48, 168):
            env[f"{col}_api_halflife_{half_life}h"] = grouped[col].transform(
                lambda s, h=half_life: s.ewm(
                    halflife=h, adjust=False, min_periods=1
                ).mean()
            )

    rolling_mean_specs = {
        "soil_moisture_0_7cm": (24, 168),
        "soil_moisture_28_100cm": (24, 168),
        "upstream_soil_moisture_0_7cm": (24, 168),
        "temperature_c": (24, 72),
        "humidity_pct": (24, 72),
        "surface_pressure_hpa": (24,),
    }
    for col, windows in rolling_mean_specs.items():
        for hours in windows:
            env[f"{col}_mean_{hours}h"] = grouped[col].transform(
                lambda s, w=hours: s.rolling(
                    w, min_periods=max(1, w // 4)
                ).mean()
            )

    # Physical sanitation. Values are not fitted against the future distribution.
    env["rainfall_mm"] = env["rainfall_mm"].clip(lower=0)
    env["rainfall_mm_max_24h"] = env["rainfall_mm_max_24h"].clip(lower=0)
    env["solar_radiation_mj_m2"] = env["solar_radiation_mj_m2"].clip(lower=0)
    for col in [c for c in env.columns if "soil_moisture" in c]:
        env[col] = env[col].clip(0, 1)
    return env


## 5. Kalender, kategori, matriks fitur, dan feature pruning

`apply_feature_pruning` di sini murni opt-in: kalau `PRUNE_FEATURES=False` atau sumber `feature_importance.csv` belum ada, seluruh fitur tetap dipakai. Baru berguna setelah minimal satu `MODE="validate"` dijalankan.


In [61]:
def add_calendar_and_categories(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    dt = out["datetime"]
    doy = dt.dt.dayofyear.astype(float)
    hour = dt.dt.hour.astype(float)
    out["dayofyear_sin"] = np.sin(2 * np.pi * doy / 365.25)
    out["dayofyear_cos"] = np.cos(2 * np.pi * doy / 365.25)
    out["hour_sin"] = np.sin(2 * np.pi * hour / 24.0)
    out["hour_cos"] = np.cos(2 * np.pi * hour / 24.0)
    out["is_wet_season"] = dt.dt.month.isin([11, 12, 1, 2, 3, 4]).astype("int8")
    out["hour_cat"] = dt.dt.hour.astype(str)
    out["month_cat"] = dt.dt.month.astype(str)
    out["mjo_phase_cat"] = out["mjo_phase"].round().astype("Int64").astype(str)
    out["landcover_class_cat"] = out["landcover_class"].astype("Int64").astype(str)
    for col in CATEGORICAL_COLUMNS:
        out[col] = out[col].fillna("__MISSING__").astype(str)
    return out


def build_feature_tables(
    train: pd.DataFrame,
    test: pd.DataFrame,
    coordinates: pd.DataFrame,
    environment: pd.DataFrame,
    paths: Paths,
    config: ModelConfig,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    rivers = _read_local_rivers(paths.rivers, coordinates)
    static = match_stations_to_rivers(coordinates, rivers)
    weights = build_upstream_weights(static, rivers, config.upstream_decay_km)
    env_features = prepare_environment(environment, weights)

    train_features = train.merge(
        env_features, on=KEYS, how="left", validate="many_to_one"
    )
    test_features = test.merge(
        env_features, on=KEYS, how="left", validate="many_to_one"
    )
    train_features = train_features.merge(
        static, on="nama_pos", how="left", validate="many_to_one"
    )
    test_features = test_features.merge(
        static, on="nama_pos", how="left", validate="many_to_one"
    )
    if (
        train_features["rainfall_mm"].isna().any()
        or test_features["rainfall_mm"].isna().any()
    ):
        raise ValueError("Some target timestamps have no matching environmental row")

    train_features = add_calendar_and_categories(train_features)
    test_features = add_calendar_and_categories(test_features)

    if config.enable_harmonic_feature:
        # Placeholder only. The real, leakage-safe values are computed per-fold
        # (fit on the train prefix only, applied to fit+valid) inside
        # run_validation / fit_final_and_submit -- never here, since here we
        # don't yet know which rows are "future" relative to a given cutoff.
        train_features["harmonic_baseline_mdpl"] = np.nan
        test_features["harmonic_baseline_mdpl"] = np.nan

    if config.enable_anchor_feature:
        # Same placeholder pattern: horizon_days is cutoff-relative (so it can
        # only be computed per-fold), and recent_anomaly_mdpl/anchor_basis_mdpl
        # depend on the fold's causal harmonic baseline residual tail.
        train_features["horizon_days"] = np.nan
        train_features["recent_anomaly_mdpl"] = np.nan
        train_features["anchor_basis_mdpl"] = np.nan
        test_features["horizon_days"] = np.nan
        test_features["recent_anomaly_mdpl"] = np.nan
        test_features["anchor_basis_mdpl"] = np.nan

    return train_features, test_features, weights


def feature_columns(frame: pd.DataFrame) -> list[str]:
    cols = [c for c in frame.columns if c not in NON_FEATURE_COLUMNS]
    # Target-derived features are intentionally forbidden.
    forbidden = [c for c in cols if c.startswith("tma_") or "target" in c.lower()]
    if forbidden:
        raise ValueError(f"Target leakage features detected: {forbidden}")
    return cols


def apply_feature_pruning(
    columns: list[str],
    importance_source: str | Path | None,
    keep_fraction: float,
) -> tuple[list[str], dict]:
    """Drop the lowest-importance features using a PRIOR run's feature_importance.csv.

    Senior review point 3: feature_importance.csv used to be computed and never
    used. Pruning is opt-in and safe by default -- if no source is given or the file
    can't be found, the full feature set is returned unchanged and the report says
    so, instead of silently changing behaviour.
    """
    report = {
        "requested": importance_source is not None,
        "applied": False,
        "kept": len(columns),
        "dropped": 0,
        "dropped_features": [],
    }
    if importance_source is None:
        return columns, report
    path = Path(importance_source)
    if not path.exists():
        LOGGER.warning("Feature importance source %s not found; keeping all features", path)
        return columns, report

    importance = pd.read_csv(path)
    # Works for both the per-fold/per-member ("fold","member","feature","importance")
    # and per-final-member ("member","family","feature","importance") schemas
    # produced by this notebook.
    agg = importance.groupby("feature", observed=True)["importance"].mean()
    agg = agg.reindex(columns).fillna(0.0)
    n_keep = max(10, int(math.ceil(len(columns) * keep_fraction)))
    kept = set(agg.sort_values(ascending=False).head(n_keep).index.tolist())
    kept_ordered = [c for c in columns if c in kept]
    dropped = sorted(set(columns) - kept)
    report.update(
        applied=True,
        kept=len(kept_ordered),
        dropped=len(dropped),
        dropped_features=dropped,
    )
    LOGGER.info(
        "Feature pruning: kept %d/%d features (dropped %d)",
        len(kept_ordered), len(columns), len(dropped),
    )
    return kept_ordered, report


## 6. Target robust, spike weighting (togglable), model builder multi-family, dan fitur baseline harmonik per pos

Target dinormalisasi robust per pos. Spike satu-titik hanya diturunkan bobotnya bila kedua tetangganya konsisten; kejadian tinggi berkelanjutan tetap berbobot penuh. `compute_sample_weights` membungkus mekanisme ini dengan toggle `use_spike_weights` agar efeknya bisa diablasi di bagian 8b, bukan sekadar diasumsikan bekerja.

Model builder sekarang mendukung dua keluarga (`catboost` dan `lightgbm`) lewat antarmuka yang sama (`build_model` / `fit_member` / `predict_member`), sehingga ensemble bisa terdiri dari anggota yang benar-benar berbeda strukturnya, bukan cuma beda seed.

**Baseline harmonik per pos (`harmonic_baseline_mdpl`)** ditambahkan sebagai fitur numerik, bukan cuma metrik pembanding. Ridge Fourier per pos di-fit kausal HANYA pada prefix training tiap fold (mirror `fit_target_stats`), lalu dievaluasi ke fit+valid pakai kalender saja -- tidak pernah membaca target baris yang diprediksi. Ini follow-up dari investigasi Wonogiri Dam: dua arsitektur berbeda total (ensemble kita dan challenger XGBoost-residual-harmonik) menunjukkan arah bias yang sama, mengindikasikan sinyal yang hilang (kemungkinan operasi bendungan), bukan masalah model -- fitur ini tidak menutup gap itu, tapi memberi trees kurva musiman/diurnal per pos yang eksplisit dan bisa diekstrapolasi 8 bulan ke depan. Toggle: `config.enable_harmonic_feature`.

**Fitur anchor (`horizon_days`, `recent_anomaly_mdpl`, `anchor_basis_mdpl`)** adalah follow-up dari hasil tim lain (station-aware stacking, leaderboard resmi 1.59656 vs punya kita 1.60232). Inti idenya: anomali per pos relatif ke baseline musiman, diukur tepat di cutoff training, dikalikan peluruhan eksponensial terhadap horizon -- ini langsung menyasar pola Wonogiri Dam (bias yang besarnya berubah tergantung seberapa jauh dari cutoff). Bedanya dengan implementasi tim lain: mereka hand-tune alpha koreksi per pos dari OOF; kita ekspos sebagai FITUR mentah ke CatBoost/LightGBM yang sudah ada, biar trees yang belajar sendiri per pos/wilayah seberapa besar harus percaya sinyal ini -- lebih sedikit parameter yang ditarik manual dari cuma 4 fold. `apply_station_features` fit satu kurva Ridge yang sama dipakai untuk kedua fitur (harmonik + anchor), bukan fit dua kali. Toggle: `config.enable_anchor_feature`.


In [62]:
def fit_target_stats(train: pd.DataFrame) -> pd.DataFrame:
    grouped = train.groupby("nama_pos", observed=True)["tma_mdpl"]
    stats = grouped.agg(
        center="median",
        q25=lambda s: s.quantile(0.25),
        q75=lambda s: s.quantile(0.75),
    )
    stats["scale"] = ((stats["q75"] - stats["q25"]) / 1.349).clip(lower=0.05)
    return stats[["center", "scale"]]


def normalize_target(frame: pd.DataFrame, stats: pd.DataFrame) -> np.ndarray:
    center = frame["nama_pos"].map(stats["center"])
    scale = frame["nama_pos"].map(stats["scale"])
    if center.isna().any() or scale.isna().any():
        raise ValueError("Target normalization encountered an unseen station")
    return ((frame["tma_mdpl"] - center) / scale).to_numpy(dtype=float)


def denormalize_prediction(
    prediction: np.ndarray, stations: pd.Series, stats: pd.DataFrame
) -> np.ndarray:
    center = stations.map(stats["center"]).to_numpy(dtype=float)
    scale = stations.map(stats["scale"]).to_numpy(dtype=float)
    return center + scale * np.asarray(prediction, dtype=float)


EARLY_STOPPING_METRICS = ("current", "rmse")


def validate_lightgbm_huber_alpha(config: ModelConfig) -> None:
    if not np.isfinite(config.lightgbm_huber_alpha):
        raise ValueError("lightgbm_huber_alpha must be finite")
    if config.lightgbm_huber_alpha <= 0:
        raise ValueError("lightgbm_huber_alpha must be positive")




def validate_early_stopping_metric(config: ModelConfig) -> None:
    if config.early_stopping_metric not in EARLY_STOPPING_METRICS:
        raise ValueError(
            "early_stopping_metric must be one of "
            f"{EARLY_STOPPING_METRICS}, got {config.early_stopping_metric!r}"
        )


MODEL_TARGET_MODES = ("raw_robust", "harmonic_residual")


def validate_model_target_config(config: ModelConfig) -> None:
    if config.target_mode not in MODEL_TARGET_MODES:
        raise ValueError(
            f"target_mode must be one of {MODEL_TARGET_MODES}, got {config.target_mode!r}"
        )
    if config.target_mode == "harmonic_residual" and not config.enable_harmonic_feature:
        raise ValueError(
            "harmonic_residual requires enable_harmonic_feature=True"
        )


def model_target_values(frame: pd.DataFrame, target_mode: str) -> np.ndarray:
    if target_mode == "raw_robust":
        values = frame["tma_mdpl"].to_numpy(dtype=float)
    elif target_mode == "harmonic_residual":
        if "harmonic_baseline_mdpl" not in frame.columns:
            raise ValueError(
                "harmonic_residual requires harmonic_baseline_mdpl; "
                "call apply_station_features on the training prefix first"
            )
        baseline = frame["harmonic_baseline_mdpl"].to_numpy(dtype=float)
        values = frame["tma_mdpl"].to_numpy(dtype=float) - baseline
    else:
        raise ValueError(f"Unknown target_mode={target_mode!r}")
    if not np.isfinite(values).all():
        raise ValueError(f"Non-finite model target in mode={target_mode}")
    return values


def fit_model_target_stats(frame: pd.DataFrame, target_mode: str) -> pd.DataFrame:
    target_frame = frame[["nama_pos"]].copy()
    target_frame["tma_mdpl"] = model_target_values(frame, target_mode)
    return fit_target_stats(target_frame)


def normalize_model_target(
    frame: pd.DataFrame, stats: pd.DataFrame, target_mode: str
) -> np.ndarray:
    center = frame["nama_pos"].map(stats["center"])
    scale = frame["nama_pos"].map(stats["scale"])
    if center.isna().any() or scale.isna().any():
        raise ValueError("Model-target normalization encountered an unseen station")
    values = model_target_values(frame, target_mode)
    normalized = (values - center.to_numpy(dtype=float)) / scale.to_numpy(dtype=float)
    if not np.isfinite(normalized).all():
        raise ValueError(f"Non-finite normalized target in mode={target_mode}")
    return normalized


def denormalize_model_prediction(
    prediction: np.ndarray,
    frame: pd.DataFrame,
    stats: pd.DataFrame,
    target_mode: str,
) -> np.ndarray:
    modeled_value = denormalize_prediction(prediction, frame["nama_pos"], stats)
    if target_mode == "harmonic_residual":
        if "harmonic_baseline_mdpl" not in frame.columns:
            raise ValueError("Missing harmonic baseline during residual inverse transform")
        modeled_value = (
            frame["harmonic_baseline_mdpl"].to_numpy(dtype=float) + modeled_value
        )
    if not np.isfinite(modeled_value).all():
        raise ValueError(f"Non-finite inverse target in mode={target_mode}")
    return np.asarray(modeled_value, dtype=float)


def target_transform_roundtrip_audit(
    frame: pd.DataFrame, stats: pd.DataFrame, target_mode: str
) -> dict:
    normalized = normalize_model_target(frame, stats, target_mode)
    restored = denormalize_model_prediction(normalized, frame, stats, target_mode)
    truth = frame["tma_mdpl"].to_numpy(dtype=float)
    absolute_error = np.abs(restored - truth)
    return {
        "target_mode": target_mode,
        "rows": int(len(frame)),
        "stations": int(frame["nama_pos"].nunique()),
        "finite_normalized": bool(np.isfinite(normalized).all()),
        "finite_restored": bool(np.isfinite(restored).all()),
        "max_roundtrip_absolute_error": float(absolute_error.max(initial=0.0)),
        "mean_roundtrip_absolute_error": float(absolute_error.mean() if len(absolute_error) else 0.0),
    }

def isolated_spike_weights(train: pd.DataFrame) -> tuple[np.ndarray, int]:
    """Downweight only one-step spikes whose two neighbors agree with each other."""
    ordered = train.sort_values(KEYS).copy()
    group = ordered.groupby("nama_pos", sort=False, observed=True)
    ordered["prev_y"] = group["tma_mdpl"].shift(1)
    ordered["next_y"] = group["tma_mdpl"].shift(-1)
    ordered["prev_t"] = group["datetime"].shift(1)
    ordered["next_t"] = group["datetime"].shift(-1)

    diff_scale = group["tma_mdpl"].transform(
        lambda s: (s.diff().abs().median() * 1.4826)
    ).clip(lower=0.05)
    station_stats = fit_target_stats(ordered)
    robust_scale = ordered["nama_pos"].map(station_stats["scale"])
    midpoint = (ordered["prev_y"] + ordered["next_y"]) / 2.0
    threshold = np.maximum(12.0 * diff_scale, 8.0 * robust_scale)
    neighbors_agree = (ordered["prev_y"] - ordered["next_y"]).abs() <= np.maximum(
        4.0 * diff_scale, 2.0 * robust_scale
    )
    adjacent = (
        ((ordered["datetime"] - ordered["prev_t"]) <= pd.Timedelta(hours=12))
        & ((ordered["next_t"] - ordered["datetime"]) <= pd.Timedelta(hours=12))
    )
    spike = adjacent & neighbors_agree & ((ordered["tma_mdpl"] - midpoint).abs() > threshold)
    ordered["weight"] = np.where(spike, 0.05, 1.0)
    weights = ordered["weight"].reindex(train.index).to_numpy(dtype=float)
    return weights, int(spike.sum())


def compute_sample_weights(train: pd.DataFrame, use_spike_weights: bool) -> tuple[np.ndarray, int]:
    """Wraps isolated_spike_weights with an ablation on/off switch.

    Senior review point 3: the spike-downweighting thresholds were never validated
    against an actual metric delta. `spike_count` is always returned (for reporting)
    even when the weights themselves are neutralized, so ablation runs stay
    comparable apples-to-apples on everything except the weighting itself.
    """
    weights, spike_count = isolated_spike_weights(train)
    if not use_spike_weights:
        weights = np.ones_like(weights)
    return weights, spike_count


def metrics_frame(y_true: np.ndarray, y_pred: np.ndarray, stations: pd.Series) -> dict:
    errors = np.abs(np.asarray(y_true) - np.asarray(y_pred))
    station_mae = pd.DataFrame({"station": stations.to_numpy(), "ae": errors}).groupby(
        "station", observed=True
    )["ae"].mean()
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(mean_squared_error(y_true, y_pred) ** 0.5),
        "macro_station_mae": float(station_mae.mean()),
        "worst_station_mae": float(station_mae.max()),
        "p95_absolute_error": float(np.quantile(errors, 0.95)),
    }


def seasonal_baseline(train: pd.DataFrame, valid: pd.DataFrame) -> np.ndarray:
    reference = train.assign(
        month=train["datetime"].dt.month,
        hour=train["datetime"].dt.hour,
    )
    lookup = reference.groupby(["nama_pos", "month", "hour"], observed=True)[
        "tma_mdpl"
    ].median()
    fallback = reference.groupby("nama_pos", observed=True)["tma_mdpl"].median()
    key = pd.MultiIndex.from_arrays(
        [valid["nama_pos"], valid["datetime"].dt.month, valid["datetime"].dt.hour]
    )
    pred = np.array(lookup.reindex(key).to_numpy(dtype=float), dtype=float, copy=True)
    missing = np.isnan(pred)
    pred[missing] = valid.loc[missing, "nama_pos"].map(fallback).to_numpy(dtype=float)
    return pred


# --------------------------------------------------------------------------------
# Per-station harmonic baseline FEATURE (not just a comparison metric).
#
# Follow-up to the Wonogiri Dam investigation: both the CatBoost/LightGBM ensemble
# and a completely different XGBoost-residual-harmonic challenger showed the SAME
# direction of bias drift across folds, which points to a missing SIGNAL (most
# likely dam operation schedule) rather than a modeling/architecture problem.
# This can't fix that gap, but it gives the tree ensemble an explicit, smoothly
# extrapolable per-station seasonal/diurnal curve fit directly against the target
# -- something month_cat/dayofyear_sin/cos alone can't provide per-station without
# the trees having to reconstruct a station x season interaction from scratch.
#
# Leakage rule: exactly like fit_target_stats, this MUST be fit only on the
# training prefix of a given fold (or the full train set for the final fit) and
# then evaluated on fit+valid (or train+test) using ONLY calendar information.
# It never reads the target value of the row it's predicting for.
# --------------------------------------------------------------------------------

def _harmonic_design_matrix(datetime_series: pd.Series) -> np.ndarray:
    doy = datetime_series.dt.dayofyear.to_numpy(dtype=float)
    hour = datetime_series.dt.hour.to_numpy(dtype=float)
    return np.column_stack([
        np.sin(2 * np.pi * doy / 365.25), np.cos(2 * np.pi * doy / 365.25),
        np.sin(4 * np.pi * doy / 365.25), np.cos(4 * np.pi * doy / 365.25),
        np.sin(2 * np.pi * hour / 24.0), np.cos(2 * np.pi * hour / 24.0),
    ])


def fit_harmonic_baselines(fit_frame: pd.DataFrame, alpha: float) -> dict[str, Ridge]:
    """One causal Ridge harmonic curve per station, fit only on `fit_frame`."""
    baselines: dict[str, Ridge] = {}
    for station, group in fit_frame.groupby("nama_pos", observed=True):
        X = _harmonic_design_matrix(group["datetime"])
        model = Ridge(alpha=alpha)
        model.fit(X, group["tma_mdpl"].to_numpy(dtype=float))
        baselines[station] = model
    return baselines


def add_harmonic_baseline_feature(
    frame: pd.DataFrame, baselines: dict[str, Ridge], fallback_center: pd.Series
) -> pd.DataFrame:
    """Overwrites the harmonic_baseline_mdpl placeholder with causal predictions.

    Stations absent from `baselines` (shouldn't normally happen since eligibility
    filtering keeps fit/valid station sets aligned, but kept as a safety net) fall
    back to their per-station median, mirroring `seasonal_baseline`'s fallback.
    """
    out = frame.copy()
    X = _harmonic_design_matrix(out["datetime"])
    pred = np.full(len(out), np.nan, dtype=float)
    stations = out["nama_pos"].to_numpy()
    for station, model in baselines.items():
        mask = stations == station
        if mask.any():
            pred[mask] = model.predict(X[mask])
    missing = np.isnan(pred)
    if missing.any():
        pred[missing] = out.loc[missing, "nama_pos"].map(fallback_center).to_numpy(dtype=float)
    out["harmonic_baseline_mdpl"] = pred
    return out


def apply_station_features(
    fit_frame: pd.DataFrame, other_frames: Sequence[pd.DataFrame], config: ModelConfig
) -> tuple[pd.DataFrame, list[pd.DataFrame]]:
    """Single entry point for every feature that needs a causal per-station
    harmonic baseline fit on `fit_frame`: harmonic_baseline_mdpl AND the
    recency/persistence "anchor" features below. The Ridge curves are fit ONCE
    per call and reused for both, instead of fitting them twice.
    Each half is independently toggle-able and safely no-ops when disabled.
    """
    if not config.enable_harmonic_feature and not config.enable_anchor_feature:
        return fit_frame, list(other_frames)

    baselines = fit_harmonic_baselines(fit_frame, config.harmonic_ridge_alpha)
    fit_out = fit_frame
    others_out = list(other_frames)

    if config.enable_harmonic_feature:
        fallback_center = fit_frame.groupby("nama_pos", observed=True)["tma_mdpl"].median()
        fit_out = add_harmonic_baseline_feature(fit_out, baselines, fallback_center)
        others_out = [add_harmonic_baseline_feature(f, baselines, fallback_center) for f in others_out]

    if config.enable_anchor_feature:
        cutoff = fit_frame["datetime"].max()
        recent_anomaly = compute_recent_anomaly(fit_frame, baselines, config.anchor_recent_window_days)
        fit_out = add_anchor_features(fit_out, cutoff, recent_anomaly, config.anchor_tau_days)
        others_out = [add_anchor_features(f, cutoff, recent_anomaly, config.anchor_tau_days) for f in others_out]

    return fit_out, others_out


# --------------------------------------------------------------------------------
# Recency/persistence "anchor" features -- follow-up to a peer team's
# station-aware stacking result (official leaderboard 1.59656 vs our 1.60232).
#
# Their key idea, distilled: per-station anomaly relative to the seasonal norm,
# measured right at the training cutoff, times an exponential horizon-decay,
# corrects persistent per-station level bias (exactly the Wonogiri Dam pattern:
# bias magnitude/direction depends on how far into the fold you are). They
# hand-tuned a per-station alpha for this correction with OOF-based shrinkage.
#
# We take the same signal but expose it as plain FEATURES fed into the existing
# CatBoost/LightGBM ensemble instead of a hand-tuned post-hoc correction -- the
# trees decide per-station/per-region how much to trust it, using the same
# leakage discipline as harmonic_baseline_mdpl (fit only on the fold's train
# prefix, evaluated causally on fit+valid).
# --------------------------------------------------------------------------------

def compute_recent_anomaly(
    fit_frame: pd.DataFrame, baselines: dict[str, Ridge], window_days: float
) -> pd.Series:
    """Per-station MEDIAN residual (tma_mdpl - harmonic_baseline) over the last
    `window_days` of `fit_frame` -- "how far off is this station right now
    relative to its normal seasonal curve, right before we start forecasting."

    Uses median rather than mean: a single sensor glitch or spike landing
    inside the 30-day recent window (we've seen these -- Napel, Bojonegoro-Kali
    Kethek, Kedungupit) would otherwise drag the mean far off, and that error
    gets fed straight into the model as a feature. Median is far more robust to
    one or two such points without needing a separate glitch detector here.
    """
    X = _harmonic_design_matrix(fit_frame["datetime"])
    baseline_pred = np.full(len(fit_frame), np.nan, dtype=float)
    stations = fit_frame["nama_pos"].to_numpy()
    for station, model in baselines.items():
        mask = stations == station
        if mask.any():
            baseline_pred[mask] = model.predict(X[mask])
    residual = fit_frame["tma_mdpl"].to_numpy(dtype=float) - baseline_pred
    cutoff = fit_frame["datetime"].max()
    window_start = cutoff - pd.Timedelta(days=window_days)
    recent_mask = (fit_frame["datetime"] >= window_start).to_numpy()
    recent = pd.DataFrame({"nama_pos": stations[recent_mask], "residual": residual[recent_mask]})
    return recent.groupby("nama_pos", observed=True)["residual"].median()


def add_anchor_features(
    frame: pd.DataFrame,
    cutoff: pd.Timestamp,
    recent_anomaly: pd.Series,
    tau_days: float,
) -> pd.DataFrame:
    out = frame.copy()
    horizon_days = (out["datetime"] - cutoff).dt.total_seconds().to_numpy() / 86_400.0
    anomaly = out["nama_pos"].map(recent_anomaly).fillna(0.0).to_numpy(dtype=float)
    out["horizon_days"] = horizon_days
    out["recent_anomaly_mdpl"] = anomaly
    out["anchor_basis_mdpl"] = np.exp(-np.abs(horizon_days) / tau_days) * anomaly
    return out


# --------------------------------------------------------------------------------
# Multi-family model builder. Every member predicts the SAME robust-normalized
# target ("z-scale"), so CatBoost and LightGBM predictions stay directly
# averageable regardless of family (senior review points 1 and 2).
# --------------------------------------------------------------------------------

def _xy_catboost(frame: pd.DataFrame, columns: Sequence[str]) -> pd.DataFrame:
    result = frame.loc[:, columns].copy()
    for col in set(columns) & set(CATEGORICAL_COLUMNS):
        result[col] = result[col].fillna("__MISSING__").astype(str)
    return result


def _xy_lightgbm(frame: pd.DataFrame, columns: Sequence[str], cat_columns: Sequence[str]) -> pd.DataFrame:
    """LightGBM wants pandas 'category' dtype rather than raw strings."""
    result = frame.loc[:, columns].copy()
    for col in cat_columns:
        result[col] = result[col].fillna("__MISSING__").astype(str).astype("category")
    return result


def make_catboost_model(member: "EnsembleMemberConfig", config: ModelConfig, iterations: int, train_dir: Path | None = None):
    from catboost import CatBoostRegressor
    params = dict(
        iterations=iterations,
        learning_rate=member.learning_rate,
        depth=member.depth,
        loss_function=f"Huber:delta={config.huber_delta}",
        eval_metric=(
            "RMSE" if config.early_stopping_metric == "rmse" else "MAE"
        ),
        l2_leaf_reg=member.l2_leaf_reg,
        random_seed=member.seed,
        random_strength=0.8,
        bootstrap_type="Bayesian",
        bagging_temperature=0.7,
        allow_writing_files=train_dir is not None,
        verbose=config.verbose_every,
    )
    if config.use_gpu:
        # GPU-friendly settings. Bayesian bootstrap (set above) is GPU-supported;
        # thread_count is a CPU-only concept and is omitted here.
        params.update(task_type="GPU", devices=config.gpu_device_id)
    else:
        params.update(task_type="CPU", thread_count=-1)
    if train_dir is not None:
        train_dir.mkdir(parents=True, exist_ok=True)
        params.update(
            train_dir=str(train_dir),
            save_snapshot=True,
            snapshot_file="training.snapshot",
            snapshot_interval=config.snapshot_interval_seconds,
        )
    return CatBoostRegressor(**params)


def make_lightgbm_model(member: "EnsembleMemberConfig", config: ModelConfig, iterations: int):
    try:
        import lightgbm as lgb
    except ImportError as exc:
        raise RuntimeError(
            "LightGBM is required for the diversified ensemble. Install requirements.txt first."
        ) from exc
    params = dict(
        objective="huber",
        alpha=config.lightgbm_huber_alpha,
        n_estimators=iterations,
        learning_rate=member.learning_rate,
        num_leaves=member.num_leaves,
        feature_fraction=member.feature_fraction,
        bagging_fraction=member.bagging_fraction,
        bagging_freq=1,
        min_child_samples=member.min_child_samples,
        random_state=member.seed,
        verbosity=-1,
    )
    if config.early_stopping_metric == "rmse":
        # Keep the Huber training objective; only align best-iteration selection
        # with the competition metric.
        params["metric"] = "rmse"
    if config.use_gpu:
        # gpu_device_id may use CatBoost's multi-device syntax ("0:1" for two
        # GPUs); LightGBM's gpu_device_id only accepts a single integer, so we
        # take the first device index. LightGBM trains one model on one GPU
        # regardless -- there's no per-model multi-GPU split like CatBoost has.
        first_device = str(config.gpu_device_id).split(":")[0].strip()
        params.update(device_type="gpu", gpu_device_id=int(first_device))
    else:
        params.update(device_type="cpu", n_jobs=-1)
    return lgb.LGBMRegressor(**params)


def build_model(member: "EnsembleMemberConfig", config: ModelConfig, iterations: int, train_dir: Path | None = None):
    if member.family == "catboost":
        return make_catboost_model(member, config, iterations, train_dir)
    if member.family == "lightgbm":
        return make_lightgbm_model(member, config, iterations)
    raise ValueError(f"Unknown model family: {member.family}")


def _fit_member_once(
    model,
    member: "EnsembleMemberConfig",
    columns: Sequence[str],
    cat_columns: Sequence[str],
    fit_frame: pd.DataFrame,
    y_fit: np.ndarray,
    weights: np.ndarray,
    valid_frame: pd.DataFrame | None = None,
    y_valid: np.ndarray | None = None,
    early_stopping_rounds: int | None = None,
):
    """Actual fit call for one ensemble member. Returns (fitted_model, best_iteration_used).
    No GPU/CPU fallback logic here -- see `fit_member` for that."""
    if member.family == "catboost":
        X_fit = _xy_catboost(fit_frame, columns)
        eval_set = None
        if valid_frame is not None:
            eval_set = (_xy_catboost(valid_frame, columns), y_valid)
        model.fit(
            X_fit, y_fit,
            cat_features=list(cat_columns),
            sample_weight=weights,
            eval_set=eval_set,
            early_stopping_rounds=early_stopping_rounds,
            use_best_model=eval_set is not None,
        )
        if eval_set is not None:
            return model, int(model.get_best_iteration()) + 1
        return model, int(model.tree_count_)

    import lightgbm as lgb
    X_fit = _xy_lightgbm(fit_frame, columns, cat_columns)
    callbacks = [lgb.log_evaluation(period=0)]
    eval_set = None
    if valid_frame is not None:
        X_valid = _xy_lightgbm(valid_frame, columns, cat_columns)
        eval_set = [(X_valid, y_valid)]
        callbacks.append(lgb.early_stopping(early_stopping_rounds, verbose=False))
    model.fit(
        X_fit, y_fit,
        sample_weight=weights,
        eval_set=eval_set,
        categorical_feature=list(cat_columns),
        callbacks=callbacks,
    )
    best_iteration = getattr(model, "best_iteration_", None)
    return model, int(best_iteration) if best_iteration else int(model.n_estimators)


def fit_member(
    model,
    member: "EnsembleMemberConfig",
    config: ModelConfig,
    iterations: int,
    columns: Sequence[str],
    cat_columns: Sequence[str],
    fit_frame: pd.DataFrame,
    y_fit: np.ndarray,
    weights: np.ndarray,
    valid_frame: pd.DataFrame | None = None,
    y_valid: np.ndarray | None = None,
    early_stopping_rounds: int | None = None,
    train_dir: Path | None = None,
):
    """Fits one ensemble member, with automatic GPU->CPU fallback.

    `model` must already have been built via `build_model(member, config, ...)`.
    If `config.use_gpu` is True and the fit raises (no GPU visible, a
    GPU-less LightGBM/CatBoost build, driver mismatch, etc.), this rebuilds the
    SAME member with `use_gpu=False` and retries once on CPU rather than
    crashing the whole run -- GPU availability can't be guaranteed across every
    Kaggle session/accelerator setting.

    Returns (fitted_model, best_iteration_used, device_used) where device_used
    is "gpu" or "cpu".
    """
    try:
        fitted, best_iteration = _fit_member_once(
            model, member, columns, cat_columns, fit_frame, y_fit, weights,
            valid_frame=valid_frame, y_valid=y_valid, early_stopping_rounds=early_stopping_rounds,
        )
        return fitted, best_iteration, ("gpu" if config.use_gpu else "cpu")
    except Exception as exc:
        if not config.use_gpu:
            raise
        LOGGER.warning(
            "GPU fit failed for member %s (%s: %s); falling back to CPU",
            member.name, type(exc).__name__, exc,
        )
        cpu_config = replace(config, use_gpu=False)
        cpu_model = build_model(member, cpu_config, iterations, train_dir)
        fitted, best_iteration = _fit_member_once(
            cpu_model, member, columns, cat_columns, fit_frame, y_fit, weights,
            valid_frame=valid_frame, y_valid=y_valid, early_stopping_rounds=early_stopping_rounds,
        )
        return fitted, best_iteration, "cpu"


def predict_member(model, member: "EnsembleMemberConfig", frame: pd.DataFrame, columns: Sequence[str], cat_columns: Sequence[str]) -> np.ndarray:
    if member.family == "catboost":
        return model.predict(_xy_catboost(frame, columns))
    return model.predict(_xy_lightgbm(frame, columns, cat_columns))


def member_feature_importance(model, member: "EnsembleMemberConfig", columns: Sequence[str]) -> dict:
    if member.family == "catboost":
        values = model.get_feature_importance()
    else:
        raw = np.asarray(model.feature_importances_, dtype=float)
        values = raw / raw.sum() * 100.0 if raw.sum() > 0 else raw
    return dict(zip(columns, (float(v) for v in values)))


## 7. Run tracking dan handoff untuk penyempurnaan model

Setiap training membuat direktori dan ZIP bertimestamp UTC. Arsip `in_progress` diperbarui selama training; arsip `complete` atau `failed` menjadi handoff akhir untuk analisis Codex.


In [63]:
HANDOFF_SCHEMA_VERSION = "1.0"


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")


def create_run_directory(output_root: Path, mode: str) -> Path:
    output_root.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    run_dir = output_root / f"handoff_{stamp}_{mode}"
    suffix = 1
    while run_dir.exists():
        run_dir = output_root / f"handoff_{stamp}_{mode}_{suffix:02d}"
        suffix += 1
    for child in ("data", "metrics", "models", "training_progress"):
        (run_dir / child).mkdir(parents=True, exist_ok=True)
    return run_dir


def update_manifest(run_dir: Path, **changes) -> dict:
    path = run_dir / "manifest.json"
    manifest = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}
    manifest.update(changes)
    manifest["updated_at_utc"] = utc_now_iso()
    write_json(path, manifest)
    return manifest


def attach_run_logger(run_dir: Path) -> None:
    for handler in list(LOGGER.handlers):
        if getattr(handler, "_tma_handoff_handler", False):
            LOGGER.removeHandler(handler)
            handler.close()
    handler = logging.FileHandler(run_dir / "run.log", encoding="utf-8")
    handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    handler._tma_handoff_handler = True
    LOGGER.addHandler(handler)
    LOGGER.setLevel(logging.INFO)


def runtime_versions() -> dict:
    packages = [
        "catboost", "lightgbm", "joblib", "geopandas", "numpy", "pandas",
        "pyarrow", "scikit-learn", "shapely",
    ]
    versions = {}
    for package in packages:
        try:
            versions[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            versions[package] = None
    return {
        "python": sys.version,
        "platform": platform.platform(),
        "packages": versions,
    }


def input_inventory(paths: Paths) -> list[dict]:
    river_sidecars = sorted(paths.rivers.parent.glob(f"{paths.rivers.stem}.*"))
    files = [paths.train, paths.test, paths.sample_submission, paths.coordinates, paths.environment, *river_sidecars]
    return [
        {"path": str(path), "exists": path.exists(), "size_bytes": path.stat().st_size if path.exists() else None}
        for path in files
    ]


def save_frame(frame: pd.DataFrame, path_without_suffix: Path) -> Path:
    """Prefer compressed parquet; fall back to gzip CSV if pyarrow is absent."""
    path_without_suffix.parent.mkdir(parents=True, exist_ok=True)
    parquet_path = path_without_suffix.with_suffix(".parquet")
    try:
        frame.to_parquet(parquet_path, index=False, compression="zstd")
        return parquet_path
    except (ImportError, ModuleNotFoundError):
        csv_path = path_without_suffix.with_suffix(".csv.gz")
        frame.to_csv(csv_path, index=False, compression="gzip")
        return csv_path


def environment_drift_report(
    train_features: pd.DataFrame, test_features: pd.DataFrame, columns: list[str]
) -> pd.DataFrame:
    numeric = [c for c in columns if pd.api.types.is_numeric_dtype(train_features[c])]
    rows = []
    for col in numeric:
        train_values = train_features[col].replace([np.inf, -np.inf], np.nan)
        test_values = test_features[col].replace([np.inf, -np.inf], np.nan)
        train_std = float(train_values.std())
        rows.append({
            "feature": col,
            "train_missing_pct": float(train_values.isna().mean() * 100),
            "test_missing_pct": float(test_values.isna().mean() * 100),
            "train_mean": float(train_values.mean()),
            "test_mean": float(test_values.mean()),
            "train_std": train_std,
            "test_std": float(test_values.std()),
            "standardized_mean_shift": float((test_values.mean() - train_values.mean()) / train_std) if train_std > 0 else np.nan,
            "train_q01": float(train_values.quantile(0.01)),
            "train_q50": float(train_values.quantile(0.50)),
            "train_q99": float(train_values.quantile(0.99)),
            "test_q01": float(test_values.quantile(0.01)),
            "test_q50": float(test_values.quantile(0.50)),
            "test_q99": float(test_values.quantile(0.99)),
        })
    return pd.DataFrame(rows)


def write_handoff_readme(run_dir: Path) -> None:
    text = """# TMA model handoff\n\nThis archive is a reproducible diagnostic handoff for the next model iteration.\n\nCore files:\n- manifest.json: run identity, timestamps, status, and stage.\n- config.json / runtime_versions.json / input_inventory.json: reproducibility context (includes the exact ensemble member configs).\n- data/train_features.* and test_features.*: exact model matrices when enabled. NOTE: if enable_harmonic_feature=True, the harmonic_baseline_mdpl column here is a NaN PLACEHOLDER -- real values are fold-specific and computed on the fly inside run_validation/fit_final_and_submit.\n- data/train_sample_diagnostics.*: normalized target, sample weights, and spike flags.\n- metrics/oof_predictions.*: leakage-safe ENSEMBLE validation predictions and residuals.\n- metrics/validation_metrics.csv: global fold scores (ensemble) and baseline comparison.\n- metrics/member_metrics.csv: per-member (CatBoost/LightGBM) fold scores, for diversity diagnostics.\n- metrics/fold_station_metrics.csv: failure analysis by station.\n- metrics/residual_slices.csv: residuals by station, month, and hour.\n- metrics/feature_importance.csv: importance by fold/member; feed this back as feature_importance_source to enable pruning.\n- metrics/environment_drift.csv: train-to-test feature distribution shift.\n- metrics/final_feature_importance.csv: importance from the final fit, per member.\n- recommended_iterations.json: median best-iteration PER ensemble member (not a single number).\n- training_progress/: live CatBoost logs and snapshots for every fold/member.\n- models/: fold and final CatBoost (.cbm) and LightGBM (.joblib) models.\n- submission.csv: exact competition output when final fit ran.\n- submission_quantiles.csv: optional CatBoost MultiQuantile prediction intervals, when enable_quantile_model=True.\n- run.log / error_traceback.txt: orchestration log and failure context.\n\nSeparately, MODE=\"ablation\" and MODE=\"tune\" write lightweight, non-zipped diagnostic\nfolders (ablation_<timestamp>/, tune_<timestamp>/) next to the handoff runs -- use\nthose to decide use_spike_weights/prune_features/hyperparameters BEFORE committing to\na full validate/fit run.\n\nUse OOF predictions—not training error—as the primary basis for refinements.\n"""
    (run_dir / "HANDOFF_README.md").write_text(text, encoding="utf-8")


def create_handoff_zip(run_dir: Path, status: str) -> Path:
    if status not in {"in_progress", "complete", "failed"}:
        raise ValueError("Invalid handoff ZIP status")
    destination = run_dir.parent / f"{run_dir.name}_{status}.zip"
    temporary = destination.with_suffix(".zip.tmp")
    if temporary.exists():
        temporary.unlink()
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
        for path in sorted(run_dir.rglob("*")):
            if path.is_file():
                already_compressed = path.suffix.lower() in {".parquet", ".gz", ".cbm", ".snapshot"}
                archive.write(
                    path, arcname=str(Path(run_dir.name) / path.relative_to(run_dir)),
                    compress_type=zipfile.ZIP_STORED if already_compressed else zipfile.ZIP_DEFLATED,
                )
    temporary.replace(destination)
    if status in {"complete", "failed"}:
        in_progress = run_dir.parent / f"{run_dir.name}_in_progress.zip"
        if in_progress.exists():
            in_progress.unlink()
    return destination


def save_pretraining_handoff(
    run_dir: Path,
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    columns: list[str],
    upstream_weights: pd.DataFrame,
    config: ModelConfig,
    save_feature_matrices: bool,
) -> None:
    cat_columns = [c for c in CATEGORICAL_COLUMNS if c in columns]
    write_json(run_dir / "feature_columns.json", {"features": columns, "categorical": cat_columns})
    upstream_weights.to_csv(run_dir / "data" / "upstream_station_weights.csv")
    stats = fit_target_stats(train_features)
    stats.to_csv(run_dir / "data" / "target_normalization.csv")
    sample_weights, _ = compute_sample_weights(train_features, config.use_spike_weights)
    diagnostics = train_features[KEYS + ["tma_mdpl"]].copy()
    diagnostics["target_normalized"] = normalize_target(train_features, stats)
    diagnostics["sample_weight"] = sample_weights
    diagnostics["isolated_spike_flag"] = (sample_weights < 1).astype("int8")
    save_frame(diagnostics, run_dir / "data" / "train_sample_diagnostics")
    drift = environment_drift_report(train_features, test_features, columns)
    drift.to_csv(run_dir / "metrics" / "environment_drift.csv", index=False)
    if save_feature_matrices:
        train_export_columns = list(dict.fromkeys(KEYS + ["tma_mdpl"] + columns))
        test_export_columns = list(dict.fromkeys(["id"] + KEYS + columns))
        train_export = train_features[train_export_columns].copy()
        test_export = test_features[test_export_columns].copy()
        save_frame(train_export, run_dir / "data" / "train_features")
        save_frame(test_export, run_dir / "data" / "test_features")
    update_manifest(run_dir, stage="features_ready", feature_count=len(columns), status="in_progress")
    create_handoff_zip(run_dir, "in_progress")


## 8. Rolling-origin validation: skor ensemble penuh per fold

Empat fold memakai horizon panjang agar menyerupai deployment 242 hari. Setiap fold melatih dan merata-ratakan **seluruh anggota ensemble** (bukan satu seed) sebelum menghitung metrik, jadi angka yang dilaporkan cocok dengan submission final. Cell ini hanya mendefinisikan fungsi; belum menjalankan training.


In [64]:
def _project_simplex(values: np.ndarray) -> np.ndarray:
    """Euclidean projection onto non-negative weights that sum to one."""
    vector = np.asarray(values, dtype=float)
    ordered = np.sort(vector)[::-1]
    cumulative = np.cumsum(ordered) - 1.0
    indices = np.arange(1, len(vector) + 1)
    positive = ordered - cumulative / indices > 0
    if not positive.any():
        return np.full(len(vector), 1.0 / len(vector))
    rho = indices[positive][-1]
    theta = cumulative[positive][-1] / rho
    projected = np.maximum(vector - theta, 0.0)
    return projected / projected.sum()


def fit_simplex_ensemble_weights(
    predictions: np.ndarray,
    target: np.ndarray,
    ridge_strength: float = 0.05,
    max_iter: int = 2000,
    tolerance: float = 1e-10,
) -> np.ndarray:
    """Fit stable non-negative RMSE weights with sum(weights)=1.

    Rows are centered by their member mean. Under the simplex constraint this
    leaves the objective unchanged while removing the large common station level,
    which substantially improves numerical conditioning. A small scale-adaptive
    ridge penalty shrinks toward equal weights when members are nearly identical.
    """
    matrix = np.asarray(predictions, dtype=float)
    y = np.asarray(target, dtype=float)
    if matrix.ndim != 2 or matrix.shape[0] != len(y):
        raise ValueError("predictions must be a rows x members matrix aligned to target")
    n_members = matrix.shape[1]
    if n_members == 1:
        return np.ones(1, dtype=float)
    finite = np.isfinite(y) & np.isfinite(matrix).all(axis=1)
    matrix, y = matrix[finite], y[finite]
    if len(y) < max(100, 10 * n_members):
        return np.full(n_members, 1.0 / n_members)

    row_mean = matrix.mean(axis=1)
    design = matrix - row_mean[:, None]
    centered_target = y - row_mean
    diversity_scale = max(float(np.mean(design ** 2)), 1e-8)
    ridge = max(0.0, ridge_strength) * diversity_scale
    equal = np.full(n_members, 1.0 / n_members)
    weights = equal.copy()
    spectral = float(np.linalg.norm(design, ord=2) ** 2)
    lipschitz = 2.0 * spectral / len(y) + 2.0 * ridge
    step = 1.0 / max(lipschitz, 1e-12)

    for _ in range(max_iter):
        residual = design @ weights - centered_target
        gradient = (
            2.0 * design.T @ residual / len(y)
            + 2.0 * ridge * (weights - equal)
        )
        updated = _project_simplex(weights - step * gradient)
        if np.max(np.abs(updated - weights)) < tolerance:
            weights = updated
            break
        weights = updated
    return weights


def _weight_mapping(
    member_names: Sequence[str], weights: np.ndarray
) -> dict[str, float]:
    return {
        name: float(weight)
        for name, weight in zip(member_names, np.asarray(weights, dtype=float))
    }


def run_validation(
    train_features: pd.DataFrame,
    columns: list[str],
    config: ModelConfig,
    artifacts: Path,
    folds: Sequence[tuple] = FOLDS,
    members: Sequence[EnsembleMemberConfig] | None = None,
    write_artifacts: bool = True,
) -> tuple[pd.DataFrame, dict[str, int]]:
    """Rolling-origin validation with leakage-safe learned ensemble weights.

    Every member prediction is retained. Fold scoring uses weights learned only
    from earlier OOF rows whose timestamps are no later than that fold's training
    cutoff. Final deployment weights are learned from the configured analogous
    September-to-May folds after validation is complete.
    """
    members = tuple(members) if members is not None else config.ensemble
    member_names = [m.name for m in members]
    equal_weights = np.full(len(members), 1.0 / len(members))
    records: list[dict] = []
    member_records: list[dict] = []
    best_iterations: dict[str, list[int]] = {m.name: [] for m in members}
    oof_parts: list[pd.DataFrame] = []
    member_oof_parts: list[pd.DataFrame] = []
    station_records: list[dict] = []
    importance_records: list[dict] = []
    blend_weight_records: list[dict] = []
    cat_columns = [c for c in CATEGORICAL_COLUMNS if c in columns]

    for fold_name, train_end, valid_start, valid_end in folds:
        train_cutoff = pd.Timestamp(train_end)
        fit = train_features[train_features["datetime"] <= train_cutoff].copy()
        valid = train_features[
            train_features["datetime"].between(
                pd.Timestamp(valid_start), pd.Timestamp(valid_end), inclusive="both"
            )
        ].copy()
        eligible = fit.groupby("nama_pos", observed=True).size().loc[lambda s: s >= 90].index
        fit = fit[fit["nama_pos"].isin(eligible)]
        valid = valid[valid["nama_pos"].isin(eligible)]
        if fit.empty or valid.empty:
            LOGGER.warning("Skipping empty fold %s", fold_name)
            continue

        validate_model_target_config(config)
        sample_weights, spike_count = compute_sample_weights(fit, config.use_spike_weights)
        # The harmonic curve is fitted only on this fold's training prefix.
        fit, (valid,) = apply_station_features(fit, (valid,), config)
        stats = fit_model_target_stats(fit, config.target_mode)
        y_fit = normalize_model_target(fit, stats, config.target_mode)
        y_valid_z = normalize_model_target(valid, stats, config.target_mode)

        member_preds_z: list[np.ndarray] = []
        member_preds_original: list[np.ndarray] = []
        fold_best_iterations: dict[str, int] = {}
        for member in members:
            progress_dir = (
                artifacts / "training_progress" / f"fold_{fold_name}_{member.name}"
                if write_artifacts else None
            )
            model = build_model(
                member, config, iterations=config.default_iterations, train_dir=progress_dir
            )
            model, best_iteration, device_used = fit_member(
                model, member, config, config.default_iterations, columns, cat_columns,
                fit, y_fit, sample_weights,
                valid_frame=valid, y_valid=y_valid_z,
                early_stopping_rounds=config.early_stopping_rounds,
                train_dir=progress_dir,
            )
            best_iterations[member.name].append(best_iteration)
            fold_best_iterations[member.name] = int(best_iteration)
            pred_z_member = predict_member(model, member, valid, columns, cat_columns)
            member_pred = denormalize_model_prediction(
                pred_z_member, valid, stats, config.target_mode
            )
            member_preds_z.append(pred_z_member)
            member_preds_original.append(member_pred)

            member_metrics = metrics_frame(
                valid["tma_mdpl"].to_numpy(), member_pred, valid["nama_pos"]
            )
            member_records.append({
                "fold": fold_name, "member": member.name, "family": member.family,
                "best_iteration": best_iteration, "device": device_used,
                **{f"model_{k}": v for k, v in member_metrics.items()},
            })

            if write_artifacts:
                for feature, importance in member_feature_importance(
                    model, member, columns
                ).items():
                    importance_records.append({
                        "fold": fold_name, "member": member.name,
                        "feature": feature, "importance": importance,
                    })
                if member.family == "catboost":
                    model.save_model(
                        str(artifacts / "models" / f"validation_{fold_name}_{member.name}.cbm")
                    )

        prediction_matrix = np.column_stack(member_preds_original)
        prediction_z_matrix = np.column_stack(member_preds_z)
        valid_target = valid["tma_mdpl"].to_numpy(dtype=float)

        # Strictly causal blend training: overlapping validation folds are cut at
        # the current fold's training cutoff before their predictions may be used.
        if config.enable_learned_ensemble_weights and member_oof_parts:
            blend_history = pd.concat(member_oof_parts, ignore_index=True)
            blend_history = blend_history[blend_history["datetime"] <= train_cutoff]
            if len(blend_history) >= 100:
                history_matrix = blend_history[member_names].to_numpy(dtype=float)
                fold_weights = fit_simplex_ensemble_weights(
                    history_matrix,
                    blend_history["tma_mdpl"].to_numpy(dtype=float),
                    ridge_strength=config.blend_ridge_strength,
                )
                weight_source = "prior_oof_before_cutoff"
            else:
                fold_weights = equal_weights
                weight_source = "equal_no_eligible_history"
        else:
            fold_weights = equal_weights
            weight_source = "equal_first_fold_or_disabled"

        pred_z = prediction_z_matrix @ fold_weights
        pred = prediction_matrix @ fold_weights
        equal_pred = prediction_matrix @ equal_weights
        model_metrics = metrics_frame(valid_target, pred, valid["nama_pos"])
        equal_metrics = metrics_frame(valid_target, equal_pred, valid["nama_pos"])
        baseline_pred = seasonal_baseline(fit, valid)
        baseline_metrics = metrics_frame(valid_target, baseline_pred, valid["nama_pos"])

        blend_weight_records.append({
            "fold": fold_name,
            "weight_source": weight_source,
            "history_rows": int(
                len(blend_history) if "blend_history" in locals() else 0
            ),
            **{f"weight_{name}": float(w) for name, w in zip(member_names, fold_weights)},
        })

        member_oof = valid[KEYS + ["tma_mdpl"]].copy()
        member_oof["fold"] = fold_name
        for name, values in zip(member_names, member_preds_original):
            member_oof[name] = values
        member_oof_parts.append(member_oof)

        if write_artifacts:
            stats_export = stats.reset_index()
            stats_export["target_mode"] = config.target_mode
            stats_export.to_csv(
                artifacts / "data" / f"target_stats_{fold_name}.csv", index=False
            )

        oof = valid[KEYS + ["tma_mdpl"]].copy()
        oof["fold"] = fold_name
        oof["target_normalized"] = y_valid_z
        oof["prediction_normalized"] = pred_z
        oof["prediction"] = pred
        oof["equal_weight_prediction"] = equal_pred
        oof["baseline_prediction"] = baseline_pred
        oof["error"] = oof["prediction"] - oof["tma_mdpl"]
        oof["absolute_error"] = oof["error"].abs()
        oof_parts.append(oof)

        for station, station_frame in oof.groupby("nama_pos", observed=True):
            station_records.append({
                "fold": fold_name, "nama_pos": station, "rows": len(station_frame),
                "mae": float(station_frame["absolute_error"].mean()),
                "rmse": float(np.sqrt(np.mean(station_frame["error"] ** 2))),
                "bias": float(station_frame["error"].mean()),
            })

        records.append({
            "fold": fold_name,
            "train_rows": len(fit), "valid_rows": len(valid),
            "valid_stations": valid["nama_pos"].nunique(),
            "isolated_spikes_downweighted": spike_count,
            "use_spike_weights": config.use_spike_weights,
            "target_mode": config.target_mode,
            "best_iteration": (
                fold_best_iterations[member_names[0]]
                if len(member_names) == 1 else np.nan
            ),
            "ensemble_members": ",".join(member_names),
            "ensemble_weight_source": weight_source,
            **{f"model_{k}": v for k, v in model_metrics.items()},
            **{f"equal_weight_{k}": v for k, v in equal_metrics.items()},
            **{f"baseline_{k}": v for k, v in baseline_metrics.items()},
        })
        LOGGER.info(
            "Completed fold %s weights=%s metrics=%s",
            fold_name, _weight_mapping(member_names, fold_weights), model_metrics,
        )

        if write_artifacts:
            save_frame(pd.concat(oof_parts, ignore_index=True), artifacts / "metrics" / "oof_predictions")
            save_frame(pd.concat(member_oof_parts, ignore_index=True), artifacts / "metrics" / "member_oof_predictions")
            pd.DataFrame(records).to_csv(artifacts / "metrics" / "validation_metrics.csv", index=False)
            pd.DataFrame(member_records).to_csv(artifacts / "metrics" / "member_metrics.csv", index=False)
            pd.DataFrame(blend_weight_records).to_csv(artifacts / "metrics" / "fold_ensemble_weights.csv", index=False)
            pd.DataFrame(station_records).to_csv(artifacts / "metrics" / "fold_station_metrics.csv", index=False)
            pd.DataFrame(importance_records).to_csv(artifacts / "metrics" / "feature_importance.csv", index=False)
            update_manifest(
                artifacts, stage=f"validation_{fold_name}_complete",
                completed_folds=[r["fold"] for r in records],
            )
            create_handoff_zip(artifacts, "in_progress")

    if not records:
        raise RuntimeError("No validation fold was runnable")
    report = pd.DataFrame(records)
    recommended = {
        name: int(np.median(iters))
        for name, iters in best_iterations.items() if iters
    }
    member_oof_all = pd.concat(member_oof_parts, ignore_index=True)
    selected = member_oof_all[
        member_oof_all["fold"].isin(config.blend_fit_folds)
    ]
    if selected.empty:
        selected = member_oof_all
        selected_folds = sorted(selected["fold"].unique().tolist())
    else:
        selected_folds = [
            fold for fold in config.blend_fit_folds
            if fold in set(selected["fold"])
        ]
    if config.enable_learned_ensemble_weights:
        final_weights = fit_simplex_ensemble_weights(
            selected[member_names].to_numpy(dtype=float),
            selected["tma_mdpl"].to_numpy(dtype=float),
            ridge_strength=config.blend_ridge_strength,
        )
    else:
        final_weights = equal_weights
    final_weight_mapping = _weight_mapping(member_names, final_weights)

    if write_artifacts:
        report.to_csv(artifacts / "metrics" / "validation_metrics.csv", index=False)
        pd.DataFrame(member_records).to_csv(artifacts / "metrics" / "member_metrics.csv", index=False)
        pd.DataFrame(blend_weight_records).to_csv(artifacts / "metrics" / "fold_ensemble_weights.csv", index=False)
        save_frame(member_oof_all, artifacts / "metrics" / "member_oof_predictions")
        oof_all = pd.concat(oof_parts, ignore_index=True)
        residual_slices = (
            oof_all.assign(month=oof_all["datetime"].dt.month, hour=oof_all["datetime"].dt.hour)
            .groupby(["nama_pos", "month", "hour"], observed=True)
            .agg(
                rows=("error", "size"), mae=("absolute_error", "mean"),
                bias=("error", "mean"),
                p95_absolute_error=("absolute_error", lambda s: s.quantile(0.95)),
            ).reset_index()
        )
        residual_slices.to_csv(artifacts / "metrics" / "residual_slices.csv", index=False)
        write_json(artifacts / "recommended_iterations.json", recommended)
        write_json(artifacts / "metrics" / "ensemble_weights.json", {
            "method": "regularized_simplex_rmse",
            "member_order": member_names,
            "final_weights": final_weight_mapping,
            "fit_folds": selected_folds,
            "fit_rows": len(selected),
            "ridge_strength": config.blend_ridge_strength,
            "target_period_api_used": False,
        })
        update_manifest(
            artifacts, stage="validation_complete",
            recommended_iterations=recommended,
            ensemble_weights=final_weight_mapping,
        )
        create_handoff_zip(artifacts, "in_progress")

    report.attrs["captured_oof_predictions"] = pd.concat(
        oof_parts, ignore_index=True
    )
    return report, recommended


## 8b. Ablasi: spike-downweighting, feature pruning, dan anchor on/off

`run_ablation_study` menjawab kritik "spike-downweighting, feature pruning, dan fitur anchor belum divalidasi secara empiris/terisolasi". Fungsi ini memakai satu model cepat (LightGBM) pada 2 fold terakhir untuk membandingkan kombinasi `use_spike_weights` on/off, fitur penuh vs. hasil pruning dari `feature_importance.csv`, DAN `anchor_on` vs `anchor_off` (harmonik + spike weighting dikunci ke setting master config, jadi satu-satunya yang beda cuma fitur anchor -- ini yang bikin efek anchor bisa diisolasi bersih, karena validasi penuh sebelumnya cenderung ganti anchor+pruning bareng sehingga susah tau kontribusi masing-masing). Jauh lebih murah daripada full ensemble x 4 fold, cukup untuk memutuskan toggle mana yang dipakai di run sesungguhnya.

**Ablasi nino_34 ON/OFF pada frozen 84 fitur** ditambahkan sebagai dua varian baru: `frozen84_nino34_on` (84 fitur lengkap, termasuk `nino_34`) dan `frozen84_nino34_off` (83 fitur: frozen 84 dikurangi `nino_34`). Keduanya dijalankan pada locked feature set yang sama persis (`ANCHOR_OFF_FROZEN_FEATURE_COLUMNS`) sehingga satu-satunya variabel yang berbeda adalah ada/tidaknya sinyal ENSO -- ini memungkinkan isolasi murni pengaruh `nino_34` tanpa confounding dari pruning atau anchor. Spike weighting dan harmonic feature dikunci ke setting master config.


In [65]:
ANCHOR_FEATURE_COLUMNS = ("horizon_days", "recent_anomaly_mdpl", "anchor_basis_mdpl")


def run_ablation_study(
    train_features: pd.DataFrame,
    columns: list[str],
    config: ModelConfig,
    artifacts: Path,
    ablation_folds: Sequence[tuple] = FOLDS[-2:],
    ablation_member: EnsembleMemberConfig | None = None,
    feature_importance_source: str | Path | None = None,
) -> pd.DataFrame:
    """Cheap on/off ablation for spike-downweighting, feature pruning, AND the
    recency/persistence anchor feature (horizon_days/recent_anomaly_mdpl/
    anchor_basis_mdpl).

    Senior review point 3: none of these should be trusted on intuition alone.
    This runs a SINGLE fast member (a LightGBM member by default) on only the
    two most recent folds -- the ones closest in length/position to the real
    test horizon -- which is cheap enough to run before committing to a full
    5-member x 4-fold validation.

    The anchor on/off variants isolate JUST the anchor's effect (harmonic
    feature and spike weighting held fixed at the master config's settings),
    which earlier full-ensemble comparisons couldn't do cleanly because
    pruning and the anchor feature were toggled together in the same run.

    Reads ``feature_importance_source`` (typically a prior run's
    ``metrics/feature_importance.csv``) to build the pruned variants; if it isn't
    available yet, only the spike-weight and anchor ablations run.
    """
    # Keep the ablation baseline identical to validate/fit.  Previously the
    # lightweight mode bypassed enforce_frozen_anchor_off_columns and trained on
    # all 99 candidates, while the confirmed anchor-OFF model used exactly 84.
    incoming_columns = list(columns)
    columns = enforce_frozen_anchor_off_columns(
        train_features, incoming_columns, config
    )
    expected_frozen_count = (
        len(ANCHOR_OFF_FROZEN_FEATURE_COLUMNS)
        if not config.enable_anchor_feature else None
    )
    parity_passed = (
        True if config.enable_anchor_feature
        else columns == list(ANCHOR_OFF_FROZEN_FEATURE_COLUMNS)
    )
    if not parity_passed:
        raise AssertionError(
            "Ablation baseline does not match the frozen anchor-OFF feature set"
        )

    member = ablation_member or next(m for m in config.ensemble if m.family == "lightgbm")
    pruned_columns, prune_report = apply_feature_pruning(
        columns, feature_importance_source, config.prune_keep_fraction
    )
    columns_no_anchor = [c for c in columns if c not in ANCHOR_FEATURE_COLUMNS]
    has_anchor_columns = len(columns_no_anchor) < len(columns)

    def _variant_config(use_spike_weights: bool, enable_anchor_feature: bool) -> ModelConfig:
        return ModelConfig(
            ensemble=config.ensemble,
            default_iterations=config.default_iterations,
            huber_delta=config.huber_delta,
            early_stopping_rounds=config.early_stopping_rounds,
            verbose_every=config.verbose_every,
            upstream_decay_km=config.upstream_decay_km,
            use_spike_weights=use_spike_weights,
            enable_harmonic_feature=config.enable_harmonic_feature,
            harmonic_ridge_alpha=config.harmonic_ridge_alpha,
            enable_anchor_feature=enable_anchor_feature,
            anchor_recent_window_days=config.anchor_recent_window_days,
            anchor_tau_days=config.anchor_tau_days,
        )

    variants = [
        {"name": "baseline_full_features_spike_on", "columns": columns,
         "use_spike_weights": True, "enable_anchor_feature": config.enable_anchor_feature},
        {"name": "no_spike_weight_full_features", "columns": columns,
         "use_spike_weights": False, "enable_anchor_feature": config.enable_anchor_feature},
    ]
    if prune_report["applied"]:
        variants += [
            {"name": "pruned_features_spike_on", "columns": pruned_columns,
             "use_spike_weights": True, "enable_anchor_feature": config.enable_anchor_feature},
            {"name": "pruned_features_no_spike", "columns": pruned_columns,
             "use_spike_weights": False, "enable_anchor_feature": config.enable_anchor_feature},
        ]
    else:
        LOGGER.warning(
            "No usable feature_importance_source; skipping pruning variants. Run "
            "MODE='validate' once, then point FEATURE_IMPORTANCE_SOURCE at the "
            "resulting metrics/feature_importance.csv to unlock pruning ablation."
        )
    if has_anchor_columns:
        # Isolated anchor comparison: harmonic feature and spike weighting held
        # fixed (at the master config's own settings) so the ONLY thing that
        # differs between these two rows is the anchor feature itself -- unlike
        # a full validate run where anchor and pruning tend to get toggled
        # together, muddying which one actually moved the metric.
        variants += [
            {"name": "anchor_on", "columns": columns,
             "use_spike_weights": config.use_spike_weights, "enable_anchor_feature": True},
            {"name": "anchor_off", "columns": columns_no_anchor,
             "use_spike_weights": config.use_spike_weights, "enable_anchor_feature": False},
        ]
    else:
        LOGGER.warning(
            "enable_anchor_feature was False when train_features was built, so anchor "
            "columns don't exist; skipping anchor on/off ablation. Set "
            "ENABLE_ANCHOR_FEATURE=True before building features to unlock this."
        )

    # Nino 3.4 ON/OFF on the frozen 84-feature set.
    # Both variants use the identical ANCHOR_OFF_FROZEN_FEATURE_COLUMNS baseline
    # (84 features, anchor OFF) so the ONLY difference is whether nino_34 is
    # included -- this isolates the pure ENSO signal contribution without any
    # confounding from pruning thresholds, anchor features, or spike weighting.
    # Spike weighting and harmonic feature are held at the master config settings.
    frozen84_columns = list(ANCHOR_OFF_FROZEN_FEATURE_COLUMNS)
    frozen84_no_nino = [c for c in frozen84_columns if c != "nino_34"]
    missing_nino34 = "nino_34" not in frozen84_columns
    if missing_nino34:
        LOGGER.warning(
            "nino_34 not found in ANCHOR_OFF_FROZEN_FEATURE_COLUMNS; "
            "skipping nino_34 ON/OFF ablation."
        )
    else:
        variants += [
            {
                "name": "frozen84_nino34_on",
                "columns": frozen84_columns,
                "use_spike_weights": config.use_spike_weights,
                "enable_anchor_feature": False,
            },
            {
                "name": "frozen84_nino34_off",
                "columns": frozen84_no_nino,
                "use_spike_weights": config.use_spike_weights,
                "enable_anchor_feature": False,
            },
        ]

    rows = []
    for variant in variants:
        variant_config = _variant_config(variant["use_spike_weights"], variant["enable_anchor_feature"])
        report, _ = run_validation(
            train_features, variant["columns"], variant_config, artifacts,
            folds=ablation_folds, members=(member,), write_artifacts=False,
        )
        keep_cols = [c for c in report.columns if c.startswith("model_") or c == "fold"]
        summary = report[keep_cols].copy()
        summary["variant"] = variant["name"]
        summary["n_features"] = len(variant["columns"])
        summary["baseline_feature_count"] = len(columns)
        summary["anchor_off_frozen_parity"] = parity_passed
        rows.append(summary)

    result = pd.concat(rows, ignore_index=True)
    artifacts.mkdir(parents=True, exist_ok=True)
    result.to_csv(artifacts / "ablation_results.csv", index=False)
    write_json(artifacts / "feature_pruning_report.json", prune_report)
    write_json(artifacts / "ablation_feature_parity_report.json", {
        "enable_anchor_feature": config.enable_anchor_feature,
        "incoming_feature_count": len(incoming_columns),
        "baseline_feature_count": len(columns),
        "expected_frozen_feature_count": expected_frozen_count,
        "parity_passed": parity_passed,
        "baseline_features": columns,
    })

    agg = result.groupby("variant", observed=True)[
        ["model_mae", "model_rmse", "model_macro_station_mae", "model_worst_station_mae"]
    ].mean()
    LOGGER.info(
        "Ablation summary using member=%s over %d fold(s):\n%s",
        member.name, len(ablation_folds), agg.sort_values("model_rmse").to_string(),
    )
    return result






def run_target_mode_ablation(
    train_features: pd.DataFrame,
    columns: list[str],
    config: ModelConfig,
    artifacts: Path,
    ablation_folds: Sequence[tuple] = FOLDS[-2:],
    ablation_member: EnsembleMemberConfig | None = None,
) -> pd.DataFrame:
    """Compare champion raw target against harmonic residual with all else locked."""
    if config.enable_anchor_feature:
        raise ValueError("target_ablation requires ENABLE_ANCHOR_FEATURE=False")
    if not config.enable_harmonic_feature:
        raise ValueError("target_ablation requires ENABLE_HARMONIC_FEATURE=True")

    columns = enforce_frozen_anchor_off_columns(train_features, columns, config)
    if columns != list(ANCHOR_OFF_FROZEN_FEATURE_COLUMNS):
        raise AssertionError("target_ablation baseline must use the exact frozen 84 features")

    member = ablation_member or next(
        m for m in config.ensemble if m.family == "lightgbm"
    )
    variants = ("raw_robust", "harmonic_residual")
    result_parts = []
    audit_rows = []

    for target_mode in variants:
        variant_config = replace(
            config,
            target_mode=target_mode,
            use_spike_weights=False,
            prune_features=False,
            enable_anchor_feature=False,
            enable_learned_ensemble_weights=False,
        )
        validate_model_target_config(variant_config)
        report, _ = run_validation(
            train_features,
            columns,
            variant_config,
            artifacts,
            folds=ablation_folds,
            members=(member,),
            write_artifacts=False,
        )
        keep = [c for c in report.columns if c.startswith("model_") or c == "fold"]
        summary = report[keep].copy()
        summary["variant"] = f"target_{target_mode}"
        summary["target_mode"] = target_mode
        summary["member"] = member.name
        summary["n_features"] = len(columns)
        summary["baseline_feature_count"] = len(columns)
        summary["anchor_off_frozen_parity"] = True
        summary["use_spike_weights"] = False
        summary["prune_features"] = False
        summary["enable_learned_ensemble_weights"] = False
        result_parts.append(summary)

        for fold_name, train_end, _, _ in ablation_folds:
            fit = train_features[
                train_features["datetime"] <= pd.Timestamp(train_end)
            ].copy()
            eligible = (
                fit.groupby("nama_pos", observed=True)
                .size().loc[lambda s: s >= 90].index
            )
            fit = fit[fit["nama_pos"].isin(eligible)]
            fit, _ = apply_station_features(fit, (), variant_config)
            stats = fit_model_target_stats(fit, target_mode)
            audit = target_transform_roundtrip_audit(fit, stats, target_mode)
            audit_rows.append({"fold": fold_name, **audit})

    result = pd.concat(result_parts, ignore_index=True)
    artifacts.mkdir(parents=True, exist_ok=True)
    result.to_csv(artifacts / "ablation_target_residual.csv", index=False)
    write_json(artifacts / "target_transform_audit.json", {
        "experiment": "raw_robust_vs_harmonic_residual",
        "member": member.name,
        "folds": [fold[0] for fold in ablation_folds],
        "feature_count_both_variants": len(columns),
        "feature_parity_passed": len(columns) == 84,
        "locked_settings": {
            "use_spike_weights": False,
            "prune_features": False,
            "enable_anchor_feature": False,
            "enable_learned_ensemble_weights": False,
            "enable_harmonic_feature": True,
            "nino_34_included": "nino_34" in columns,
        },
        "only_intended_difference": "target_mode",
        "target_period_api_used": False,
        "roundtrip_checks": audit_rows,
    })
    LOGGER.info(
        "Target-mode ablation using member=%s:\n%s",
        member.name,
        result[["fold", "variant", "model_rmse", "model_mae", "model_worst_station_mae"]].to_string(index=False),
    )
    return result

def run_metric_ablation(
    train_features: pd.DataFrame,
    columns: list[str],
    config: ModelConfig,
    artifacts: Path,
    ablation_folds: Sequence[tuple] = FOLDS[-2:],
) -> pd.DataFrame:
    """Compare legacy early stopping against RMSE for CatBoost and LightGBM.

    The objective remains Huber. Only the validation metric used to select the
    best iteration changes. All champion data/feature/ensemble toggles are
    locked so this is a clean metric-alignment experiment.
    """
    if config.enable_anchor_feature:
        raise ValueError("metric_ablation requires ENABLE_ANCHOR_FEATURE=False")
    if config.target_mode != "raw_robust":
        raise ValueError("metric_ablation requires TARGET_MODE='raw_robust'")
    if not config.enable_harmonic_feature:
        raise ValueError("metric_ablation requires ENABLE_HARMONIC_FEATURE=True")

    columns = enforce_frozen_anchor_off_columns(train_features, columns, config)
    if columns != list(ANCHOR_OFF_FROZEN_FEATURE_COLUMNS):
        raise AssertionError("metric_ablation must use the exact frozen 84 features")

    cat_member = next(m for m in config.ensemble if m.family == "catboost")
    lgbm_member = next(m for m in config.ensemble if m.family == "lightgbm")
    members = (cat_member, lgbm_member)
    metric_variants = ("current", "rmse")
    result_parts = []

    for metric_variant in metric_variants:
        variant_config = replace(
            config,
            early_stopping_metric=metric_variant,
            target_mode="raw_robust",
            use_spike_weights=False,
            prune_features=False,
            enable_anchor_feature=False,
            enable_learned_ensemble_weights=False,
        )
        validate_early_stopping_metric(variant_config)
        for member in members:
            report, _ = run_validation(
                train_features,
                columns,
                variant_config,
                artifacts,
                folds=ablation_folds,
                members=(member,),
                write_artifacts=False,
            )
            keep = [
                c for c in report.columns
                if c.startswith("model_") or c in {"fold", "target_mode"}
            ]
            summary = report[keep].copy()
            summary["variant"] = f"early_stop_{metric_variant}"
            summary["early_stopping_metric"] = metric_variant
            summary["member"] = member.name
            summary["family"] = member.family
            summary["n_features"] = len(columns)
            summary["baseline_feature_count"] = len(columns)
            summary["anchor_off_frozen_parity"] = True
            summary["use_spike_weights"] = False
            summary["prune_features"] = False
            summary["enable_learned_ensemble_weights"] = False
            result_parts.append(summary)

    result = pd.concat(result_parts, ignore_index=True)
    artifacts.mkdir(parents=True, exist_ok=True)
    result.to_csv(
        artifacts / "ablation_early_stopping_metric.csv", index=False
    )
    write_json(artifacts / "metric_ablation_audit.json", {
        "experiment": "legacy_validation_metric_vs_rmse",
        "objective_locked": "huber",
        "members": [
            {"name": member.name, "family": member.family}
            for member in members
        ],
        "folds": [fold[0] for fold in ablation_folds],
        "feature_count_both_variants": len(columns),
        "feature_parity_passed": len(columns) == 84,
        "metric_mapping": {
            "current": {
                "catboost": "MAE",
                "lightgbm": "objective default (Huber)",
            },
            "rmse": {"catboost": "RMSE", "lightgbm": "rmse"},
        },
        "locked_settings": {
            "target_mode": "raw_robust",
            "use_spike_weights": False,
            "prune_features": False,
            "enable_anchor_feature": False,
            "enable_learned_ensemble_weights": False,
            "enable_harmonic_feature": True,
            "nino_34_included": "nino_34" in columns,
        },
        "only_intended_difference": "early_stopping_metric",
        "target_period_api_used": False,
    })
    LOGGER.info(
        "Early-stopping metric ablation:\n%s",
        result[[
            "fold", "family", "variant", "model_rmse", "model_mae",
            "model_worst_station_mae",
        ]].to_string(index=False),
    )
    return result

def run_lgbm_alpha_ablation(
    train_features: pd.DataFrame,
    columns: list[str],
    config: ModelConfig,
    artifacts: Path,
    alpha_values: Sequence[float] = (0.7, 0.9, 1.2, 1.5),
    ablation_folds: Sequence[tuple] = FOLDS[-2:],
    ablation_member: EnsembleMemberConfig | None = None,
) -> pd.DataFrame:
    """Tune only LightGBM's Huber alpha; CatBoost delta is untouched."""
    if config.enable_anchor_feature:
        raise ValueError("lgbm_alpha_ablation requires ENABLE_ANCHOR_FEATURE=False")
    if config.target_mode != "raw_robust":
        raise ValueError("lgbm_alpha_ablation requires TARGET_MODE='raw_robust'")
    if config.early_stopping_metric != "current":
        raise ValueError(
            "lgbm_alpha_ablation requires EARLY_STOPPING_METRIC='current'"
        )
    if not config.enable_harmonic_feature:
        raise ValueError("lgbm_alpha_ablation requires harmonic feature ON")

    columns = enforce_frozen_anchor_off_columns(train_features, columns, config)
    if columns != list(ANCHOR_OFF_FROZEN_FEATURE_COLUMNS):
        raise AssertionError(
            "lgbm_alpha_ablation must use the exact frozen 84 features"
        )

    member = ablation_member or next(
        m for m in config.ensemble if m.name == "lgbm_a"
    )
    if member.family != "lightgbm":
        raise ValueError("lgbm_alpha_ablation member must be LightGBM")

    alpha_values = tuple(float(value) for value in alpha_values)
    if not alpha_values or any(value <= 0 for value in alpha_values):
        raise ValueError("Every LightGBM Huber alpha must be positive")
    if len(set(alpha_values)) != len(alpha_values):
        raise ValueError("LightGBM Huber alpha grid contains duplicates")

    result_parts = []
    for alpha in alpha_values:
        variant_config = replace(
            config,
            lightgbm_huber_alpha=alpha,
            early_stopping_metric="current",
            target_mode="raw_robust",
            use_spike_weights=False,
            prune_features=False,
            enable_anchor_feature=False,
            enable_learned_ensemble_weights=False,
        )
        validate_lightgbm_huber_alpha(variant_config)
        report, _ = run_validation(
            train_features,
            columns,
            variant_config,
            artifacts,
            folds=ablation_folds,
            members=(member,),
            write_artifacts=False,
        )
        keep = [
            c for c in report.columns
            if c.startswith("model_")
            or c in {"fold", "target_mode", "best_iteration"}
        ]
        summary = report[keep].copy()
        summary["variant"] = f"lgbm_huber_alpha_{alpha:g}"
        summary["lightgbm_huber_alpha"] = alpha
        summary["catboost_huber_delta_unchanged"] = config.huber_delta
        summary["early_stopping_metric"] = "current"
        summary["member"] = member.name
        summary["family"] = member.family
        summary["n_features"] = len(columns)
        summary["baseline_feature_count"] = len(columns)
        summary["anchor_off_frozen_parity"] = True
        summary["use_spike_weights"] = False
        summary["prune_features"] = False
        summary["enable_learned_ensemble_weights"] = False
        result_parts.append(summary)

    result = pd.concat(result_parts, ignore_index=True)
    artifacts.mkdir(parents=True, exist_ok=True)
    result.to_csv(
        artifacts / "ablation_lightgbm_huber_alpha.csv", index=False
    )
    write_json(artifacts / "lgbm_alpha_ablation_audit.json", {
        "experiment": "lightgbm_huber_alpha_grid",
        "objective_locked": "huber",
        "member": {"name": member.name, "family": member.family},
        "alpha_values": list(alpha_values),
        "baseline_alpha": 1.5,
        "catboost_huber_delta_unchanged": config.huber_delta,
        "folds": [fold[0] for fold in ablation_folds],
        "feature_count_all_variants": len(columns),
        "feature_parity_passed": len(columns) == 84,
        "locked_settings": {
            "target_mode": "raw_robust",
            "early_stopping_metric": "current",
            "use_spike_weights": False,
            "prune_features": False,
            "enable_anchor_feature": False,
            "enable_learned_ensemble_weights": False,
            "enable_harmonic_feature": True,
            "nino_34_included": "nino_34" in columns,
        },
        "only_intended_difference": "lightgbm_huber_alpha",
        "best_iteration_recorded_per_fold": "best_iteration" in result.columns,
        "target_period_api_used": False,
    })
    LOGGER.info(
        "LightGBM Huber-alpha ablation:\n%s",
        result[[
            "fold", "lightgbm_huber_alpha", "best_iteration",
            "model_rmse", "model_mae", "model_worst_station_mae",
        ]].to_string(index=False),
    )
    return result

def run_lgbm_alpha_confirmation(
    train_features: pd.DataFrame,
    columns: list[str],
    config: ModelConfig,
    artifacts: Path,
    alpha_values: tuple[float, float] = (0.7, 1.5),
    confirmation_fold_names: tuple[str, ...] = (
        "sep_2023", "sep_2024", "jan_2025"
    ),
) -> pd.DataFrame:
    """Confirm alpha 0.7 and its 50:50 blend against alpha 1.5.

    Runs lgbm_a and lgbm_b independently. Predictions from both alpha values
    are retained in memory and blended row-by-row, so the blend requires no
    additional fitted model and is evaluated on exactly aligned OOF rows.
    """
    if config.enable_anchor_feature:
        raise ValueError(
            "lgbm_alpha_confirmation requires ENABLE_ANCHOR_FEATURE=False"
        )
    if config.target_mode != "raw_robust":
        raise ValueError(
            "lgbm_alpha_confirmation requires TARGET_MODE='raw_robust'"
        )
    if config.early_stopping_metric != "current":
        raise ValueError(
            "lgbm_alpha_confirmation requires EARLY_STOPPING_METRIC='current'"
        )
    if tuple(float(value) for value in alpha_values) != (0.7, 1.5):
        raise ValueError("Confirmation alpha values must be exactly (0.7, 1.5)")

    fold_lookup = {fold[0]: fold for fold in FOLDS}
    missing_folds = sorted(set(confirmation_fold_names) - set(fold_lookup))
    if missing_folds:
        raise ValueError(f"Unknown confirmation folds: {missing_folds}")
    confirmation_folds = tuple(
        fold_lookup[name] for name in confirmation_fold_names
    )

    columns = enforce_frozen_anchor_off_columns(train_features, columns, config)
    if columns != list(ANCHOR_OFF_FROZEN_FEATURE_COLUMNS):
        raise AssertionError(
            "lgbm_alpha_confirmation must use exact frozen 84 features"
        )

    members = tuple(
        member for member in config.ensemble
        if member.name in {"lgbm_a", "lgbm_b"}
    )
    if {member.name for member in members} != {"lgbm_a", "lgbm_b"}:
        raise ValueError("Both lgbm_a and lgbm_b are required for confirmation")

    result_parts = []
    predictions_by_member_alpha: dict[tuple[str, float], pd.DataFrame] = {}
    key_columns = ["fold", *KEYS]

    for member in members:
        for alpha in alpha_values:
            variant_config = replace(
                config,
                lightgbm_huber_alpha=float(alpha),
                early_stopping_metric="current",
                target_mode="raw_robust",
                use_spike_weights=False,
                prune_features=False,
                enable_anchor_feature=False,
                enable_learned_ensemble_weights=False,
            )
            report, _ = run_validation(
                train_features,
                columns,
                variant_config,
                artifacts,
                folds=confirmation_folds,
                members=(member,),
                write_artifacts=False,
            )
            captured = report.attrs.get("captured_oof_predictions")
            if captured is None or captured.empty:
                raise RuntimeError(
                    "run_validation did not return captured OOF predictions"
                )
            prediction_frame = captured[
                key_columns + ["tma_mdpl", "prediction"]
            ].copy()
            if prediction_frame.duplicated(key_columns).any():
                raise AssertionError("Duplicate OOF keys in alpha confirmation")
            predictions_by_member_alpha[(member.name, float(alpha))] = (
                prediction_frame.rename(columns={"prediction": f"prediction_alpha_{alpha:g}"})
            )

            keep = [
                c for c in report.columns
                if c.startswith("model_")
                or c in {"fold", "target_mode", "best_iteration"}
            ]
            summary = report[keep].copy()
            summary["strategy"] = f"alpha_{alpha:g}"
            summary["variant"] = f"{member.name}_alpha_{alpha:g}"
            summary["lightgbm_huber_alpha"] = float(alpha)
            summary["member"] = member.name
            summary["family"] = member.family
            summary["n_features"] = len(columns)
            summary["anchor_off_frozen_parity"] = True
            result_parts.append(summary)

    oof_exports = []
    for member in members:
        alpha_07 = predictions_by_member_alpha[(member.name, 0.7)]
        alpha_15 = predictions_by_member_alpha[(member.name, 1.5)]
        merged = alpha_07.merge(
            alpha_15,
            on=key_columns + ["tma_mdpl"],
            how="inner",
            validate="one_to_one",
        )
        if len(merged) != len(alpha_07) or len(merged) != len(alpha_15):
            raise AssertionError("OOF rows do not align between alpha variants")
        merged["prediction_blend_50_50"] = 0.5 * (
            merged["prediction_alpha_0.7"]
            + merged["prediction_alpha_1.5"]
        )
        merged["member"] = member.name
        oof_exports.append(merged)

        for fold_name, fold_frame in merged.groupby("fold", observed=True):
            blend_metrics = metrics_frame(
                fold_frame["tma_mdpl"].to_numpy(dtype=float),
                fold_frame["prediction_blend_50_50"].to_numpy(dtype=float),
                fold_frame["nama_pos"],
            )
            result_parts.append(pd.DataFrame([{
                "fold": fold_name,
                "target_mode": "raw_robust",
                "best_iteration": np.nan,
                **{f"model_{key}": value for key, value in blend_metrics.items()},
                "strategy": "blend_50_50",
                "variant": f"{member.name}_blend_alpha_0.7_1.5",
                "lightgbm_huber_alpha": np.nan,
                "member": member.name,
                "family": member.family,
                "n_features": len(columns),
                "anchor_off_frozen_parity": True,
            }]))

    result = pd.concat(result_parts, ignore_index=True)
    strategy_order = {"alpha_0.7": 0, "blend_50_50": 1, "alpha_1.5": 2}
    result["_strategy_order"] = result["strategy"].map(strategy_order)
    result = result.sort_values(
        ["member", "fold", "_strategy_order"]
    ).drop(columns="_strategy_order").reset_index(drop=True)

    baseline = result[result["strategy"] == "alpha_1.5"][[
        "member", "fold", "model_rmse", "model_worst_station_mae"
    ]].rename(columns={
        "model_rmse": "baseline_rmse",
        "model_worst_station_mae": "baseline_worst_station_mae",
    })

    gate_results = {}
    exact_folds = {"sep_2023", "sep_2024"}
    for strategy in ("alpha_0.7", "blend_50_50"):
        candidate = result[result["strategy"] == strategy].merge(
            baseline, on=["member", "fold"], how="left", validate="one_to_one"
        )
        candidate["rmse_delta_pct"] = 100.0 * (
            candidate["model_rmse"] / candidate["baseline_rmse"] - 1.0
        )
        candidate["worst_station_delta_pct"] = 100.0 * (
            candidate["model_worst_station_mae"]
            / candidate["baseline_worst_station_mae"] - 1.0
        )
        exact = candidate[candidate["fold"].isin(exact_folds)]
        safety = candidate[candidate["fold"] == "jan_2025"]
        exact_wins = int((exact["rmse_delta_pct"] < 0).sum())
        exact_mean_improvement_pct = float(-exact["rmse_delta_pct"].mean())
        jan_max_rmse_degradation_pct = float(safety["rmse_delta_pct"].max())
        jan_max_worst_degradation_pct = float(
            safety["worst_station_delta_pct"].max()
        )
        passed = bool(
            exact_wins >= 3
            and exact_mean_improvement_pct >= 0.5
            and jan_max_rmse_degradation_pct <= 0.3
            and jan_max_worst_degradation_pct <= 2.0
        )
        gate_results[strategy] = {
            "exact_season_wins_out_of_4": exact_wins,
            "exact_season_mean_rmse_improvement_pct": exact_mean_improvement_pct,
            "jan_2025_max_rmse_degradation_pct": jan_max_rmse_degradation_pct,
            "jan_2025_max_worst_station_degradation_pct": jan_max_worst_degradation_pct,
            "passed": passed,
        }

    passing = [
        strategy for strategy, values in gate_results.items()
        if values["passed"]
    ]
    promoted_strategy = (
        max(
            passing,
            key=lambda strategy: gate_results[strategy][
                "exact_season_mean_rmse_improvement_pct"
            ],
        )
        if passing else None
    )

    artifacts.mkdir(parents=True, exist_ok=True)
    result.to_csv(
        artifacts / "lgbm_alpha_confirmation_results.csv", index=False
    )
    save_frame(
        pd.concat(oof_exports, ignore_index=True),
        artifacts / "lgbm_alpha_confirmation_oof",
    )
    write_json(artifacts / "lgbm_alpha_confirmation_audit.json", {
        "experiment": "alpha_0.7_vs_1.5_and_oof_blend",
        "members": [member.name for member in members],
        "folds": list(confirmation_fold_names),
        "exact_season_folds": sorted(exact_folds),
        "alpha_values": list(alpha_values),
        "blend": {"alpha_0.7": 0.5, "alpha_1.5": 0.5},
        "feature_count_all_variants": len(columns),
        "feature_parity_passed": len(columns) == 84,
        "locked_settings": {
            "target_mode": "raw_robust",
            "early_stopping_metric": "current",
            "use_spike_weights": False,
            "prune_features": False,
            "enable_anchor_feature": False,
            "enable_learned_ensemble_weights": False,
            "enable_harmonic_feature": True,
            "nino_34_included": "nino_34" in columns,
        },
        "oof_alignment_keys": key_columns,
        "target_period_api_used": False,
    })
    write_json(artifacts / "lgbm_alpha_confirmation_decision.json", {
        "acceptance_criteria": {
            "exact_season_wins_minimum": 3,
            "exact_season_combinations": 4,
            "exact_season_mean_rmse_improvement_minimum_pct": 0.5,
            "jan_2025_rmse_degradation_maximum_pct": 0.3,
            "jan_2025_worst_station_degradation_maximum_pct": 2.0,
        },
        "strategies": gate_results,
        "promoted_strategy": promoted_strategy,
        "full_validation_authorized": promoted_strategy is not None,
    })
    LOGGER.info(
        "LightGBM alpha confirmation decision: %s",
        json.dumps({
            "strategies": gate_results,
            "promoted_strategy": promoted_strategy,
        }, indent=2),
    )
    return result

# Frozen after the four-fold/full-ensemble anchor confirmation. This is the
# confirmed 87-feature model matrix with only the three anchor columns removed.
# Do not recompute this list from an 85% fraction when anchor is disabled: 85%
# of the 99 remaining candidates would select 85 columns, not this exact set.
ANCHOR_OFF_FROZEN_FEATURE_COLUMNS = ('nama_pos', 'rainfall_mm', 'humidity_pct', 'dew_point_c', 'cloud_cover_pct', 'temperature_c', 'wind_speed_kmh', 'rainfall_max_24h_mm', 'solar_radiation_mj_m2', 'soil_moisture_0_7cm', 'soil_moisture_7_28cm', 'soil_moisture_28_100cm', 'soil_moisture_100_255cm', 'surface_pressure_hpa', 'pressure_msl_hpa', 'built_surface_m2', 'landcover_name', 'rmm1', 'rmm2', 'mjo_amplitude', 'mjo_active', 'nino_34', 'wind_direction_sin', 'wind_direction_cos', 'dewpoint_depression_c', 'upstream_rainfall_mm', 'upstream_soil_moisture_0_7cm', 'rainfall_mm_sum_6h', 'rainfall_mm_sum_24h', 'rainfall_mm_sum_72h', 'rainfall_mm_sum_168h', 'rainfall_mm_sum_336h', 'rainfall_mm_max_24h', 'rainfall_mm_api_halflife_12h', 'rainfall_mm_api_halflife_48h', 'rainfall_mm_api_halflife_168h', 'upstream_rainfall_mm_sum_6h', 'upstream_rainfall_mm_sum_24h', 'upstream_rainfall_mm_sum_72h', 'upstream_rainfall_mm_sum_168h', 'upstream_rainfall_mm_sum_336h', 'upstream_rainfall_mm_max_24h', 'upstream_rainfall_mm_api_halflife_12h', 'upstream_rainfall_mm_api_halflife_48h', 'upstream_rainfall_mm_api_halflife_168h', 'soil_moisture_0_7cm_mean_24h', 'soil_moisture_0_7cm_mean_168h', 'soil_moisture_28_100cm_mean_24h', 'soil_moisture_28_100cm_mean_168h', 'upstream_soil_moisture_0_7cm_mean_24h', 'upstream_soil_moisture_0_7cm_mean_168h', 'temperature_c_mean_24h', 'temperature_c_mean_72h', 'humidity_pct_mean_24h', 'humidity_pct_mean_72h', 'surface_pressure_hpa_mean_24h', 'latitude', 'longitude', 'river_distance_m', 'LENGTH_KM', 'DIST_DN_KM', 'DIST_UP_KM', 'CATCH_SKM', 'UPLAND_SKM', 'DIS_AV_CMS', 'ORD_STRA', 'ORD_CLAS', 'ORD_FLOW', 'river_match_confidence', 'log1p_catch_skm', 'log1p_upland_skm', 'log1p_dis_av_cms', 'log1p_length_km', 'main_riv_cat', 'hybas_l12_cat', 'dayofyear_sin', 'dayofyear_cos', 'hour_sin', 'is_wet_season', 'hour_cat', 'month_cat', 'mjo_phase_cat', 'landcover_class_cat', 'harmonic_baseline_mdpl')


def enforce_frozen_anchor_off_columns(
    frame: pd.DataFrame,
    candidate_columns: Sequence[str],
    config: ModelConfig,
) -> list[str]:
    """Return the exact confirmed 84-feature set when anchor is disabled."""
    if config.enable_anchor_feature:
        return list(candidate_columns)

    frozen = list(ANCHOR_OFF_FROZEN_FEATURE_COLUMNS)
    missing = sorted(set(frozen) - set(frame.columns))
    leaked_anchor = sorted(set(frozen) & set(ANCHOR_FEATURE_COLUMNS))
    if missing:
        raise ValueError(f"Frozen anchor-off features missing from matrix: {missing}")
    if leaked_anchor:
        raise AssertionError(f"Anchor features leaked into frozen set: {leaked_anchor}")
    if len(frozen) != 84 or len(set(frozen)) != 84:
        raise AssertionError("Frozen anchor-off feature list must contain 84 unique columns")
    LOGGER.info(
        "Anchor OFF: overriding dynamic pruning with confirmed frozen %d-feature set",
        len(frozen),
    )
    return frozen


## 8c. Eksperimen P0 senior: pseudo-origin anchor dan missing radiasi

Cell ini menambahkan dua eksperimen yang **tidak mengubah perilaku mode final**:

1. **Pseudo-origin anchor** — contoh training dibentuk dari beberapa forecast origin historis. Setiap contoh hanya memakai target sebelum origin untuk menghitung harmonic baseline dan recent anomaly, lalu memprediksi horizon positif sampai 242 hari. Ini menghapus train-serving mismatch pada implementasi anchor lama.
2. **Audit missing radiasi** — pola missing radiasi test direplikasi sebagai tail-block pada validation. Empat strategi dibandingkan: native missing, drop radiasi, imputasi climatology, dan climatology + masked augmentation.

Gunakan `MODE = "p0_experiments"`. Default eksperimen memakai satu LightGBM cepat dan dua fold terbaru agar murah. Setelah varian menang, konfirmasi dengan empat fold dan ensemble penuh sebelum dipromosikan ke final pipeline. Artefak yang dihasilkan:

- `p0_anchor_summary.csv` dan `p0_anchor_horizon_metrics.csv`;
- `p0_solar_summary.csv` dan `p0_solar_horizon_metrics.csv`;
- `p0_experiment_config.json`.

API target periode submission tidak dibaca oleh eksperimen ini.


In [ ]:
P0_HORIZON_BUCKETS = (-1e-9, 30.0, 90.0, 180.0, 242.0, np.inf)
P0_HORIZON_LABELS = ("0-30", "31-90", "91-180", "181-242", ">242")


def _p0_fast_member(config: ModelConfig) -> EnsembleMemberConfig:
    """Use the strongest cheap LightGBM member unless explicitly overridden."""
    lightgbm_members = [m for m in config.ensemble if m.family == "lightgbm"]
    if not lightgbm_members:
        raise ValueError("P0 experiments require at least one LightGBM member")
    return next((m for m in lightgbm_members if m.name == "lgbm_b"), lightgbm_members[-1])


def _horizon_metric_rows(
    frame: pd.DataFrame,
    prediction: np.ndarray,
    fold: str,
    variant: str,
    experiment: str,
    origin: pd.Timestamp,
    evaluation: str = "validation",
) -> list[dict]:
    horizon = (frame["datetime"] - origin).dt.total_seconds() / 86_400.0
    bucket = pd.cut(
        horizon,
        bins=P0_HORIZON_BUCKETS,
        labels=P0_HORIZON_LABELS,
        include_lowest=True,
        right=True,
    )
    scored = pd.DataFrame(
        {
            "tma_mdpl": frame["tma_mdpl"].to_numpy(dtype=float),
            "prediction": np.asarray(prediction, dtype=float),
            "horizon_bucket": bucket,
        }
    )
    rows = []
    for label, group in scored.groupby("horizon_bucket", observed=True):
        metric = metrics_frame(
            group["tma_mdpl"].to_numpy(),
            group["prediction"].to_numpy(),
            pd.Series(np.repeat("all", len(group))),
        )
        rows.append(
            {
                "experiment": experiment,
                "fold": fold,
                "variant": variant,
                "evaluation": evaluation,
                "horizon_bucket": str(label),
                "rows": len(group),
                "mae": metric["mae"],
                "rmse": metric["rmse"],
                "bias": float((group["prediction"] - group["tma_mdpl"]).mean()),
                "p95_absolute_error": metric["p95_absolute_error"],
            }
        )
    return rows


def _absolute_error_slope_per_30d(
    frame: pd.DataFrame, prediction: np.ndarray, origin: pd.Timestamp
) -> float:
    """Linear degradation rate of absolute error in metres per 30 lead days."""
    horizon = (
        (frame["datetime"] - origin).dt.total_seconds().to_numpy(dtype=float)
        / 86_400.0
    )
    absolute_error = np.abs(
        np.asarray(prediction, dtype=float)
        - frame["tma_mdpl"].to_numpy(dtype=float)
    )
    finite = np.isfinite(horizon) & np.isfinite(absolute_error)
    if finite.sum() < 2 or np.ptp(horizon[finite]) == 0:
        return float("nan")
    return float(np.polyfit(horizon[finite], absolute_error[finite], 1)[0] * 30.0)


def build_pseudo_origin_training_frame(
    fit_frame: pd.DataFrame,
    config: ModelConfig,
    stride_days: int = 60,
    min_history_days: int = 180,
    max_horizon_days: int = 242,
) -> pd.DataFrame:
    """Expand a fold's fit prefix into causal positive-horizon training examples.

    For every historical origin, harmonic models and the 30-day recent anomaly
    are fitted strictly from rows at or before that origin. Labels are taken only
    after the origin, up to ``max_horizon_days``. ``_source_position`` preserves
    the original row identity so robust spike weights can be replicated safely.
    """
    if stride_days <= 0 or min_history_days <= 0 or max_horizon_days <= 0:
        raise ValueError("Pseudo-origin day parameters must be positive")
    if not config.enable_anchor_feature:
        raise ValueError("Pseudo-origin experiment requires enable_anchor_feature=True")

    # Assign source positions before sorting: sample weights are aligned to the
    # incoming fit-frame order and must survive the chronological reordering.
    base = (
        fit_frame.reset_index(drop=True)
        .assign(_source_position=lambda x: np.arange(len(x), dtype=np.int64))
        .sort_values(KEYS)
        .reset_index(drop=True)
    )
    first_origin = (base["datetime"].min() + pd.Timedelta(days=min_history_days)).normalize()
    first_origin += pd.Timedelta(hours=18)
    last_origin = base["datetime"].max() - pd.Timedelta(hours=6)
    origins = pd.date_range(first_origin, last_origin, freq=f"{stride_days}D")
    parts: list[pd.DataFrame] = []

    for origin in origins:
        history = base[base["datetime"] <= origin]
        eligible = history.groupby("nama_pos", observed=True).size().loc[lambda s: s >= 90].index
        history = history[history["nama_pos"].isin(eligible)]
        future = base[
            (base["datetime"] > origin)
            & (base["datetime"] <= origin + pd.Timedelta(days=max_horizon_days))
            & base["nama_pos"].isin(eligible)
        ].copy()
        if history.empty or future.empty:
            continue

        baselines = fit_harmonic_baselines(history, config.harmonic_ridge_alpha)
        fallback = history.groupby("nama_pos", observed=True)["tma_mdpl"].median()
        if config.enable_harmonic_feature:
            future = add_harmonic_baseline_feature(future, baselines, fallback)
        recent = compute_recent_anomaly(
            history, baselines, config.anchor_recent_window_days
        )
        future = add_anchor_features(
            future, origin, recent, config.anchor_tau_days
        )
        future["_pseudo_origin"] = origin
        parts.append(future)

    if not parts:
        raise RuntimeError("No pseudo-origin training examples could be constructed")
    expanded = pd.concat(parts, ignore_index=True)
    if (expanded["horizon_days"] <= 0).any():
        raise AssertionError("Pseudo-origin training produced non-positive horizon_days")
    return expanded


def _fit_one_p0_member(
    fit_frame: pd.DataFrame,
    valid_frame: pd.DataFrame,
    columns: Sequence[str],
    member: EnsembleMemberConfig,
    config: ModelConfig,
    y_fit: np.ndarray,
    y_valid: np.ndarray,
    weights: np.ndarray,
):
    cat_columns = [c for c in CATEGORICAL_COLUMNS if c in columns]
    model = build_model(member, config, config.default_iterations)
    model, best_iteration, device = fit_member(
        model,
        member,
        config,
        config.default_iterations,
        columns,
        cat_columns,
        fit_frame,
        y_fit,
        weights,
        valid_frame=valid_frame,
        y_valid=y_valid,
        early_stopping_rounds=config.early_stopping_rounds,
    )
    return model, best_iteration, device, cat_columns


def run_pseudo_origin_anchor_experiment(
    train_features: pd.DataFrame,
    columns: list[str],
    config: ModelConfig,
    artifacts: Path,
    folds: Sequence[tuple] = FOLDS[-2:],
    member: EnsembleMemberConfig | None = None,
    stride_days: int = 60,
    min_history_days: int = 180,
    max_horizon_days: int = 242,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Compare legacy cutoff anchor with causal pseudo-origin anchor."""
    missing_anchor = [c for c in ANCHOR_FEATURE_COLUMNS if c not in columns]
    if missing_anchor:
        raise ValueError(f"P0 anchor experiment needs columns: {missing_anchor}")
    member = member or _p0_fast_member(config)
    rows: list[dict] = []
    horizon_rows: list[dict] = []

    for fold_name, train_end, valid_start, valid_end in folds:
        fit = train_features[train_features["datetime"] <= pd.Timestamp(train_end)].copy()
        valid = train_features[
            train_features["datetime"].between(
                pd.Timestamp(valid_start), pd.Timestamp(valid_end), inclusive="both"
            )
        ].copy()
        eligible = fit.groupby("nama_pos", observed=True).size().loc[lambda s: s >= 90].index
        fit = fit[fit["nama_pos"].isin(eligible)].copy()
        valid = valid[valid["nama_pos"].isin(eligible)].copy()
        stats = fit_target_stats(fit)
        y_valid = normalize_target(valid, stats)
        base_weights, _ = compute_sample_weights(fit, config.use_spike_weights)

        legacy_fit, (legacy_valid,) = apply_station_features(fit, (valid,), config)
        pseudo_fit = build_pseudo_origin_training_frame(
            fit,
            config,
            stride_days=stride_days,
            min_history_days=min_history_days,
            max_horizon_days=max_horizon_days,
        )
        # Validation is always served from the real fold origin. Reuse the same
        # validation matrix for both variants so only training construction differs.
        pseudo_valid = legacy_valid

        variants = (
            (
                "legacy_cutoff_anchor",
                legacy_fit,
                normalize_target(legacy_fit, stats),
                base_weights,
            ),
            (
                "pseudo_origin_anchor",
                pseudo_fit,
                normalize_target(pseudo_fit, stats),
                base_weights[pseudo_fit["_source_position"].to_numpy(dtype=int)],
            ),
        )
        for variant, fit_variant, y_fit, weights in variants:
            model, best_iteration, device, cat_columns = _fit_one_p0_member(
                fit_variant,
                pseudo_valid,
                columns,
                member,
                config,
                y_fit,
                y_valid,
                weights,
            )
            pred_z = predict_member(model, member, pseudo_valid, columns, cat_columns)
            pred = denormalize_prediction(pred_z, pseudo_valid["nama_pos"], stats)
            metric = metrics_frame(
                pseudo_valid["tma_mdpl"].to_numpy(), pred, pseudo_valid["nama_pos"]
            )
            rows.append(
                {
                    "fold": fold_name,
                    "variant": variant,
                    "member": member.name,
                    "train_rows": len(fit_variant),
                    "valid_rows": len(pseudo_valid),
                    "best_iteration": best_iteration,
                    "device": device,
                    **{f"model_{k}": v for k, v in metric.items()},
                    "model_bias": float(
                        np.mean(pred - pseudo_valid["tma_mdpl"].to_numpy(dtype=float))
                    ),
                    "absolute_error_slope_per_30d": _absolute_error_slope_per_30d(
                        pseudo_valid, pred, pd.Timestamp(train_end)
                    ),
                }
            )
            horizon_rows.extend(
                _horizon_metric_rows(
                    pseudo_valid,
                    pred,
                    fold_name,
                    variant,
                    "pseudo_origin_anchor",
                    pd.Timestamp(train_end),
                )
            )

    summary = pd.DataFrame(rows)
    horizon = pd.DataFrame(horizon_rows)
    summary.to_csv(artifacts / "p0_anchor_summary.csv", index=False)
    horizon.to_csv(artifacts / "p0_anchor_horizon_metrics.csv", index=False)
    return summary, horizon


def _solar_test_missing_profile(test_features: pd.DataFrame) -> dict:
    solar = test_features[["datetime", "solar_radiation_mj_m2"]].copy()
    by_time = solar.groupby("datetime", observed=True)["solar_radiation_mj_m2"].apply(
        lambda s: bool(s.isna().any())
    )
    missing_times = by_time[by_time].index
    if len(missing_times) == 0:
        raise ValueError("Test features contain no missing solar radiation to emulate")
    start, end = by_time.index.min(), by_time.index.max()
    first_missing = missing_times.min()
    fraction = (first_missing - start).total_seconds() / max(
        (end - start).total_seconds(), 1.0
    )
    return {
        "test_missing_cell_fraction": float(solar["solar_radiation_mj_m2"].isna().mean()),
        "first_missing_datetime": str(first_missing),
        "tail_start_fraction": float(fraction),
    }


def _mask_validation_solar_tail(
    frame: pd.DataFrame, tail_start_fraction: float
) -> tuple[pd.DataFrame, np.ndarray]:
    out = frame.copy()
    start, end = out["datetime"].min(), out["datetime"].max()
    elapsed = (out["datetime"] - start).dt.total_seconds()
    duration = max((end - start).total_seconds(), 1.0)
    mask = (elapsed / duration >= tail_start_fraction).to_numpy()
    out.loc[mask, "solar_radiation_mj_m2"] = np.nan
    out["solar_radiation_missing"] = mask.astype("int8")
    return out, mask


def _solar_climatology(fit_frame: pd.DataFrame) -> pd.Series:
    reference = fit_frame.assign(
        month=fit_frame["datetime"].dt.month,
        hour=fit_frame["datetime"].dt.hour,
    )
    return reference.groupby(
        ["nama_pos", "month", "hour"], observed=True
    )["solar_radiation_mj_m2"].median()


def _impute_solar_from_climatology(
    frame: pd.DataFrame, climatology: pd.Series
) -> pd.DataFrame:
    out = frame.copy()
    key = pd.MultiIndex.from_arrays(
        [out["nama_pos"], out["datetime"].dt.month, out["datetime"].dt.hour]
    )
    fill = climatology.reindex(key).to_numpy(dtype=float)
    missing = out["solar_radiation_mj_m2"].isna().to_numpy()
    out.loc[missing, "solar_radiation_mj_m2"] = fill[missing]
    if out["solar_radiation_mj_m2"].isna().any():
        fallback = float(climatology.median())
        out["solar_radiation_mj_m2"] = out["solar_radiation_mj_m2"].fillna(fallback)
    return out


def run_solar_missing_experiment(
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    columns: list[str],
    config: ModelConfig,
    artifacts: Path,
    folds: Sequence[tuple] = FOLDS[-2:],
    member: EnsembleMemberConfig | None = None,
    augmentation_fraction: float = 0.5,
) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    """Stress-test solar strategies using the test missing-tail geometry."""
    if "solar_radiation_mj_m2" not in columns:
        raise ValueError("Solar experiment requires solar_radiation_mj_m2 in columns")
    if not 0 < augmentation_fraction <= 1:
        raise ValueError("augmentation_fraction must be in (0, 1]")
    member = member or _p0_fast_member(config)
    profile = _solar_test_missing_profile(test_features)
    rows: list[dict] = []
    horizon_rows: list[dict] = []

    for fold_name, train_end, valid_start, valid_end in folds:
        fit = train_features[train_features["datetime"] <= pd.Timestamp(train_end)].copy()
        valid = train_features[
            train_features["datetime"].between(
                pd.Timestamp(valid_start), pd.Timestamp(valid_end), inclusive="both"
            )
        ].copy()
        eligible = fit.groupby("nama_pos", observed=True).size().loc[lambda s: s >= 90].index
        fit = fit[fit["nama_pos"].isin(eligible)].copy()
        valid = valid[valid["nama_pos"].isin(eligible)].copy()
        stats = fit_target_stats(fit)
        base_weights, _ = compute_sample_weights(fit, config.use_spike_weights)
        fit, (valid,) = apply_station_features(fit, (valid,), config)
        y_fit_base = normalize_target(fit, stats)
        y_valid = normalize_target(valid, stats)
        climatology = _solar_climatology(fit)
        valid_stress, stress_mask = _mask_validation_solar_tail(
            valid, profile["tail_start_fraction"]
        )

        variants = []
        native_fit = fit.copy()
        native_valid = valid.copy()
        native_stress = valid_stress.copy()
        if "solar_radiation_missing" in columns:
            native_fit["solar_radiation_missing"] = 0
            native_valid["solar_radiation_missing"] = 0
        variants.append(
            ("native_missing", native_fit, native_valid, native_stress, columns, y_fit_base, base_weights)
        )

        drop_columns = [
            c for c in columns
            if c not in {"solar_radiation_mj_m2", "solar_radiation_missing"}
        ]
        variants.append(
            ("drop_solar", fit, valid, valid_stress, drop_columns, y_fit_base, base_weights)
        )

        impute_columns = list(columns)
        if "solar_radiation_missing" not in impute_columns:
            impute_columns.append("solar_radiation_missing")
        climate_fit = fit.copy()
        climate_fit["solar_radiation_missing"] = 0
        climate_valid = valid.copy()
        climate_valid["solar_radiation_missing"] = 0
        climate_stress = _impute_solar_from_climatology(valid_stress, climatology)
        variants.append(
            (
                "climatology_imputation",
                climate_fit,
                climate_valid,
                climate_stress,
                impute_columns,
                y_fit_base,
                base_weights,
            )
        )

        # Deterministic augmentation: duplicate a stable subset with solar hidden
        # and replaced by station-month-hour climatology. Original rows are retained.
        selector = (
            pd.util.hash_pandas_object(
                fit[["datetime", "nama_pos"]], index=False
            ).to_numpy(dtype=np.uint64)
            / np.float64(np.iinfo(np.uint64).max)
            < augmentation_fraction
        )
        augmented = fit.loc[selector].copy()
        augmented["solar_radiation_mj_m2"] = np.nan
        augmented["solar_radiation_missing"] = 1
        augmented = _impute_solar_from_climatology(augmented, climatology)
        original = fit.copy()
        original["solar_radiation_missing"] = 0
        augmentation_fit = pd.concat([original, augmented], ignore_index=True)
        augmentation_y = np.concatenate([y_fit_base, y_fit_base[selector]])
        augmentation_weights = np.concatenate([base_weights, base_weights[selector]])
        variants.append(
            (
                "masked_augmentation",
                augmentation_fit,
                climate_valid,
                climate_stress,
                impute_columns,
                augmentation_y,
                augmentation_weights,
            )
        )

        for variant, fit_variant, valid_normal, valid_masked, variant_columns, y_fit, weights in variants:
            model, best_iteration, device, cat_columns = _fit_one_p0_member(
                fit_variant,
                valid_normal,
                variant_columns,
                member,
                config,
                y_fit,
                y_valid,
                weights,
            )
            for evaluation, evaluated in (
                ("normal", valid_normal),
                ("masked_tail", valid_masked),
            ):
                pred_z = predict_member(
                    model, member, evaluated, variant_columns, cat_columns
                )
                pred = denormalize_prediction(pred_z, evaluated["nama_pos"], stats)
                metric = metrics_frame(
                    evaluated["tma_mdpl"].to_numpy(), pred, evaluated["nama_pos"]
                )
                rows.append(
                    {
                        "fold": fold_name,
                        "variant": variant,
                        "evaluation": evaluation,
                        "member": member.name,
                        "n_features": len(variant_columns),
                        "train_rows": len(fit_variant),
                        "valid_rows": len(evaluated),
                        "masked_valid_rows": int(stress_mask.sum()),
                        "best_iteration": best_iteration,
                        "device": device,
                        **{f"model_{k}": v for k, v in metric.items()},
                        "model_bias": float(
                            np.mean(pred - evaluated["tma_mdpl"].to_numpy(dtype=float))
                        ),
                        "absolute_error_slope_per_30d": _absolute_error_slope_per_30d(
                            evaluated, pred, pd.Timestamp(train_end)
                        ),
                    }
                )
                horizon_rows.extend(
                    _horizon_metric_rows(
                        evaluated,
                        pred,
                        fold_name,
                        variant,
                        "solar_missing",
                        pd.Timestamp(train_end),
                        evaluation=evaluation,
                    )
                )

    summary = pd.DataFrame(rows)
    horizon = pd.DataFrame(horizon_rows)
    summary.to_csv(artifacts / "p0_solar_summary.csv", index=False)
    horizon.to_csv(artifacts / "p0_solar_horizon_metrics.csv", index=False)
    return summary, horizon, profile


def run_p0_experiments(
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    columns: list[str],
    config: ModelConfig,
    artifacts: Path,
    folds: Sequence[tuple] = FOLDS[-2:],
    member: EnsembleMemberConfig | None = None,
    pseudo_stride_days: int = 60,
    pseudo_min_history_days: int = 180,
    pseudo_max_horizon_days: int = 242,
    solar_augmentation_fraction: float = 0.5,
) -> dict:
    """Run both senior-priority P0 experiments and persist auditable tables."""
    artifacts.mkdir(parents=True, exist_ok=True)
    member = member or _p0_fast_member(config)
    anchor_summary, anchor_horizon = run_pseudo_origin_anchor_experiment(
        train_features,
        columns,
        config,
        artifacts,
        folds=folds,
        member=member,
        stride_days=pseudo_stride_days,
        min_history_days=pseudo_min_history_days,
        max_horizon_days=pseudo_max_horizon_days,
    )
    solar_summary, solar_horizon, solar_profile = run_solar_missing_experiment(
        train_features,
        test_features,
        columns,
        config,
        artifacts,
        folds=folds,
        member=member,
        augmentation_fraction=solar_augmentation_fraction,
    )
    experiment_config = {
        "member": asdict(member),
        "folds": list(folds),
        "shared_feature_count": len(columns),
        "pseudo_stride_days": pseudo_stride_days,
        "pseudo_min_history_days": pseudo_min_history_days,
        "pseudo_max_horizon_days": pseudo_max_horizon_days,
        "solar_augmentation_fraction": solar_augmentation_fraction,
        "solar_test_missing_profile": solar_profile,
        "target_period_api_used": False,
    }
    write_json(artifacts / "p0_experiment_config.json", experiment_config)
    return {
        "anchor_summary": anchor_summary,
        "anchor_horizon": anchor_horizon,
        "solar_summary": solar_summary,
        "solar_horizon": solar_horizon,
        "solar_profile": solar_profile,
        "experiment_dir": str(artifacts),
    }


## 8d. Hyperparameter sweep ringan (CatBoost)

`run_hyperparam_sweep` menjawab kritik "depth=8, learning_rate=0.035, l2_leaf_reg=10.0 tidak punya jejak tuning". Bukan pengganti search penuh — cuma menguji beberapa kombinasi depth/learning_rate/l2 pada fold `jan_2025` (paling mirip panjang/posisi kalender dengan horizon test asli) supaya keputusan hyperparameter berbasis angka, bukan tebakan.


In [66]:
def run_hyperparam_sweep(
    train_features: pd.DataFrame,
    columns: list[str],
    config: ModelConfig,
    sweep_fold: tuple = FOLDS[-1],
    grid: Sequence[dict] | None = None,
) -> pd.DataFrame:
    """Lightweight, single-fold CatBoost hyperparameter sanity check.

    Senior review point 4: `depth=8, learning_rate=0.035, l2_leaf_reg=10.0` had no
    documented search behind them. This is not a substitute for a proper search --
    it tests a handful of depth/learning_rate/l2 combinations on the fold most
    representative of the real test horizon (``jan_2025`` by default: same length
    and calendar position as the actual 19 Sep 2025 - 18 Mei 2026 test window) so a
    hyperparameter decision is backed by a number instead of a guess.
    """
    grid = grid or [
        {"depth": 6, "learning_rate": 0.05, "l2_leaf_reg": 6.0},
        {"depth": 8, "learning_rate": 0.035, "l2_leaf_reg": 10.0},
        {"depth": 8, "learning_rate": 0.02, "l2_leaf_reg": 10.0},
        {"depth": 10, "learning_rate": 0.025, "l2_leaf_reg": 14.0},
    ]
    fold_name, train_end, valid_start, valid_end = sweep_fold
    fit = train_features[train_features["datetime"] <= pd.Timestamp(train_end)].copy()
    valid = train_features[
        train_features["datetime"].between(
            pd.Timestamp(valid_start), pd.Timestamp(valid_end), inclusive="both"
        )
    ].copy()
    eligible = fit.groupby("nama_pos", observed=True).size().loc[lambda s: s >= 90].index
    fit = fit[fit["nama_pos"].isin(eligible)]
    valid = valid[valid["nama_pos"].isin(eligible)]
    if fit.empty or valid.empty:
        raise RuntimeError(f"Sweep fold {fold_name} has no data; check FOLDS/sweep_fold")

    validate_model_target_config(config)
    weights, _ = compute_sample_weights(fit, config.use_spike_weights)
    fit, (valid,) = apply_station_features(fit, (valid,), config)
    stats = fit_model_target_stats(fit, config.target_mode)
    y_fit = normalize_model_target(fit, stats, config.target_mode)
    y_valid_z = normalize_model_target(valid, stats, config.target_mode)
    cat_columns = [c for c in CATEGORICAL_COLUMNS if c in columns]

    rows = []
    for params in grid:
        member = EnsembleMemberConfig(name="sweep", family="catboost", seed=17, **params)
        model = build_model(member, config, iterations=config.default_iterations, train_dir=None)
        model, best_iteration, device_used = fit_member(
            model, member, config, config.default_iterations, columns, cat_columns, fit, y_fit, weights,
            valid_frame=valid, y_valid=y_valid_z,
            early_stopping_rounds=config.early_stopping_rounds,
        )
        pred = denormalize_model_prediction(
            predict_member(model, member, valid, columns, cat_columns),
            valid, stats, config.target_mode,
        )
        metrics = metrics_frame(valid["tma_mdpl"].to_numpy(), pred, valid["nama_pos"])
        rows.append({"fold": fold_name, **params, "best_iteration": best_iteration, "device": device_used, **metrics})
        LOGGER.info("Sweep %s: %s", params, metrics)

    result = pd.DataFrame(rows).sort_values("rmse").reset_index(drop=True)
    return result


## 9. Final ensemble fit, submission, dan model quantile opsional

Setiap anggota ensemble dilatih dengan jumlah iterasi rekomendasinya sendiri (median hasil validasi per member, bukan satu angka global), lalu prediksi dirata-ratakan pada skala target ternormalisasi. Bila `enable_quantile_model=True`, satu CatBoost `MultiQuantile` tambahan dilatih untuk interval prediksi (`submission_quantiles.csv`) di luar titik prediksi utama. Cell ini hanya mendefinisikan fungsi.


In [67]:
def fit_quantile_model(
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    columns: list[str],
    cat_columns: list[str],
    config: ModelConfig,
    artifacts: Path,
    weights: np.ndarray,
    y: np.ndarray,
    target_stats: pd.DataFrame,
) -> Path:
    """Optional CatBoost MultiQuantile model.

    Senior review point 4: point predictions alone don't communicate risk for a
    flood-relevant forecast. This trains one extra CatBoost model with
    ``MultiQuantile`` loss on the SAME robust-normalized target as the main
    ensemble, and writes a prediction-interval CSV alongside ``submission.csv``.
    Off by default (``config.enable_quantile_model``); it roughly doubles the final
    training time of one CatBoost member.
    """
    from catboost import CatBoostRegressor

    def _build_quantile_model(use_gpu: bool):
        alpha = ",".join(str(q) for q in config.quantile_levels)
        params = dict(
            iterations=config.default_iterations,
            learning_rate=0.03,
            depth=8,
            loss_function=f"MultiQuantile:alpha={alpha}",
            random_seed=97,
            l2_leaf_reg=10.0,
            verbose=config.verbose_every,
        )
        if use_gpu:
            params.update(task_type="GPU", devices=config.gpu_device_id)
        else:
            params.update(task_type="CPU", thread_count=-1)
        return CatBoostRegressor(**params)

    model = _build_quantile_model(config.use_gpu)
    try:
        model.fit(_xy_catboost(train_features, columns), y, cat_features=cat_columns, sample_weight=weights)
    except Exception as exc:
        if not config.use_gpu:
            raise
        LOGGER.warning(
            "GPU fit failed for quantile model (%s: %s); falling back to CPU", type(exc).__name__, exc
        )
        model = _build_quantile_model(False)
        model.fit(_xy_catboost(train_features, columns), y, cat_features=cat_columns, sample_weight=weights)
    pred_z = np.asarray(model.predict(_xy_catboost(test_features, columns)))
    if pred_z.ndim == 1:
        pred_z = pred_z.reshape(-1, len(config.quantile_levels))

    quantile_frame = test_features[["id"]].copy()
    for i, q in enumerate(config.quantile_levels):
        quantile_frame[f"tma_mdpl_q{int(round(q * 100)):02d}"] = denormalize_model_prediction(
            pred_z[:, i], test_features, target_stats, config.target_mode
        )

    output_path = artifacts / "submission_quantiles.csv"
    quantile_frame.to_csv(output_path, index=False)
    model.save_model(str(artifacts / "models" / "catboost_quantile.cbm"))
    return output_path


def fit_final_and_submit(
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    sample: pd.DataFrame,
    columns: list[str],
    config: ModelConfig,
    artifacts: Path,
    iterations_by_member: dict[str, int],
    ensemble_weights: dict[str, float] | None = None,
) -> Path:
    artifacts.mkdir(parents=True, exist_ok=True)
    validate_model_target_config(config)
    weights, spike_count = compute_sample_weights(train_features, config.use_spike_weights)
    train_features, (test_features,) = apply_station_features(train_features, (test_features,), config)
    stats = fit_model_target_stats(train_features, config.target_mode)
    y = normalize_model_target(train_features, stats, config.target_mode)
    cat_columns = [c for c in CATEGORICAL_COLUMNS if c in columns]
    predictions = []
    final_importance_records = []
    devices_used: dict[str, str] = {}

    for member in config.ensemble:
        iterations = iterations_by_member.get(member.name, config.default_iterations)
        progress_dir = artifacts / "training_progress" / f"final_{member.name}"
        model = build_model(
            member, config, iterations=iterations,
            train_dir=progress_dir if member.family == "catboost" else None,
        )
        model, _, device_used = fit_member(
            model, member, config, iterations, columns, cat_columns, train_features, y, weights,
            train_dir=progress_dir if member.family == "catboost" else None,
        )
        devices_used[member.name] = device_used

        if member.family == "catboost":
            model.save_model(str(artifacts / "models" / f"catboost_{member.name}.cbm"))
        else:
            import joblib
            joblib.dump(model, artifacts / "models" / f"lightgbm_{member.name}.joblib")

        predictions.append(predict_member(model, member, test_features, columns, cat_columns))
        for feature, importance in member_feature_importance(model, member, columns).items():
            final_importance_records.append({
                "member": member.name, "family": member.family, "feature": feature, "importance": importance,
            })
        pd.DataFrame(final_importance_records).to_csv(
            artifacts / "metrics" / "final_feature_importance.csv", index=False
        )
        completed = [
            m.name for m in config.ensemble
            if (artifacts / "models" / f"catboost_{m.name}.cbm").exists()
            or (artifacts / "models" / f"lightgbm_{m.name}.joblib").exists()
        ]
        update_manifest(artifacts, stage=f"final_{member.name}_complete", completed_final_members=completed)
        create_handoff_zip(artifacts, "in_progress")

    if ensemble_weights:
        raw_weights = np.array(
            [ensemble_weights.get(member.name, 0.0) for member in config.ensemble],
            dtype=float,
        )
        if (raw_weights < 0).any() or raw_weights.sum() <= 0:
            raise ValueError("Invalid learned ensemble weights")
        final_weights = raw_weights / raw_weights.sum()
    else:
        final_weights = np.full(len(config.ensemble), 1.0 / len(config.ensemble))
    mean_z = np.column_stack(predictions) @ final_weights
    prediction = denormalize_model_prediction(
        mean_z, test_features, stats, config.target_mode
    )

    quantile_path = None
    if config.enable_quantile_model:
        quantile_path = fit_quantile_model(
            train_features, test_features, columns, cat_columns, config, artifacts, weights, y, stats
        )

    ordered = pd.DataFrame(
        {
            "_row_order": test_features["_row_order"],
            "id": test_features["id"],
            "tma_mdpl": prediction,
        }
    ).sort_values("_row_order")
    submission = sample[["id"]].merge(
        ordered[["id", "tma_mdpl"]], on="id", how="left", validate="one_to_one"
    )
    if submission["tma_mdpl"].isna().any() or not np.isfinite(submission["tma_mdpl"]).all():
        raise ValueError("Submission contains missing/non-finite predictions")
    output_path = artifacts / "submission.csv"
    submission.to_csv(output_path, index=False)

    stats_export = stats.copy()
    stats_export["target_mode"] = config.target_mode
    stats_export.to_csv(artifacts / "data" / "target_normalization.csv")
    (artifacts / "feature_columns.json").write_text(
        json.dumps({"features": columns, "categorical": cat_columns}, indent=2),
        encoding="utf-8",
    )
    metadata = {
        "iterations_by_member": iterations_by_member,
        "devices_used": devices_used,
        "requested_use_gpu": config.use_gpu,
        "ensemble": [asdict(m) for m in config.ensemble],
        "use_spike_weights": config.use_spike_weights,
        "target_mode": config.target_mode,
        "isolated_spikes_downweighted": spike_count,
        "train_rows": len(train_features),
        "test_rows": len(test_features),
        "quantile_output": str(quantile_path) if quantile_path else None,
        "ensemble_weights": _weight_mapping(
            [member.name for member in config.ensemble], final_weights
        ),
    }
    (artifacts / "run_metadata.json").write_text(
        json.dumps(metadata, indent=2), encoding="utf-8"
    )
    update_manifest(artifacts, stage="final_fit_complete", final_iterations_by_member=iterations_by_member)
    create_handoff_zip(artifacts, "in_progress")
    return output_path


## 10. Orkestrator notebook dan mode eksekusi

`run_pipeline` sekarang punya 6 mode: `audit`, `tune`, `validate`, `ablation`, `fit`, `all`. Mode `tune` dan `ablation` menulis folder diagnostik ringan (`tune_<timestamp>/`, `ablation_<timestamp>/`) tanpa ZIP handoff besar — dipakai untuk memutuskan konfigurasi sebelum commit ke `validate`/`fit`/`all` yang penuh.


In [68]:
def find_latest_ensemble_weights(output_dir: Path) -> dict[str, float] | None:
    candidates = sorted(
        Path(output_dir).glob("handoff_*/metrics/ensemble_weights.json"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    for path in candidates:
        try:
            payload = json.loads(path.read_text(encoding="utf-8"))
            weights = payload.get("final_weights", {})
            if weights:
                return {str(k): float(v) for k, v in weights.items()}
        except (OSError, ValueError, TypeError, json.JSONDecodeError):
            continue
    return None


def find_latest_recommended_iterations(output_dir: Path) -> dict[str, int] | None:
    candidates = sorted(
        output_dir.glob("handoff_*/recommended_iterations.json"),
        key=lambda path: path.stat().st_mtime, reverse=True,
    )
    if not candidates:
        return None
    return json.loads(candidates[0].read_text(encoding="utf-8"))


def run_pipeline(
    mode: str = "audit",
    data_dir: Path = Path("datasets"),
    output_dir: Path = Path("artifacts"),
    config: ModelConfig | None = None,
    iterations_by_member: dict[str, int] | None = None,
    save_feature_matrices: bool = True,
) -> dict:
    """Run a Kaggle/local mode and create a timestamped diagnostic handoff.

    Modes:
      audit     - input validation only, no training.
      ablation  - cheap on/off ablation of spike-weighting / feature-pruning.
                  Writes ablation_<timestamp>/ablation_results.csv, no ZIP.
      target_ablation  - raw robust target versus leakage-safe harmonic residual.
                  Writes target_ablation_<timestamp>/ablation_target_residual.csv.
      metric_ablation  - legacy validation metric versus RMSE on CatBoost + LightGBM.
                  Writes metric_ablation_<timestamp>/ablation_early_stopping_metric.csv.
      lgbm_alpha_ablation  - LightGBM Huber-alpha grid with all champion settings locked.
                  Writes lgbm_alpha_ablation_<timestamp>/ablation_lightgbm_huber_alpha.csv.
      lgbm_alpha_confirmation  - confirm alpha 0.7, alpha 1.5, and their OOF blend.
                  Writes confirmation metrics, aligned OOF predictions, and an automatic gate.
      tune      - light single-fold CatBoost hyperparameter sweep.
                  Writes tune_<timestamp>/hyperparam_sweep.csv, no ZIP.
      p0_experiments - pseudo-origin anchor + solar missing stress tests.
                  Writes p0_experiments_<timestamp>/*.csv, no ZIP.
      p0_experiments - pseudo-origin anchor + solar missing stress tests.
                  Writes p0_experiments_<timestamp>/*.csv, no ZIP.
      p0_experiments - pseudo-origin anchor + solar missing stress tests.
                  Writes p0_experiments_<timestamp>/*.csv, no ZIP.
      p0_experiments - pseudo-origin anchor + solar missing stress tests.
                  Writes p0_experiments_<timestamp>/*.csv, no ZIP.
      p0_experiments - pseudo-origin anchor + solar missing stress tests.
                  Writes p0_experiments_<timestamp>/*.csv, no ZIP.
      p0_experiments - pseudo-origin anchor + solar missing stress tests.
                  Writes p0_experiments_<timestamp>/*.csv, no ZIP.
      p0_experiments - pseudo-origin anchor + solar missing stress tests.
                  Writes p0_experiments_<timestamp>/*.csv, no ZIP.
      p0_experiments - pseudo-origin anchor + solar missing stress tests.
                  Writes p0_experiments_<timestamp>/*.csv, no ZIP.
      p0_experiments - pseudo-origin anchor + solar missing stress tests.
                  Writes p0_experiments_<timestamp>/*.csv, no ZIP.
      validate  - full ensemble rolling-origin validation, no final fit.
      fit       - final ensemble fit + submission using the iterations passed in,
                  or the latest previously recommended per-member iterations.
      all       - validate, then fit, then a complete handoff ZIP.

    Run order recommendation: audit -> tune -> validate -> ablation
    (now that feature_importance.csv exists) -> fit/all with the chosen config.
    """
    if mode not in {"audit", "ablation", "target_ablation", "metric_ablation", "lgbm_alpha_ablation", "lgbm_alpha_confirmation", "tune", "p0_experiments", "validate", "fit", "all"}:
        raise ValueError("mode must be one of: audit, ablation, target_ablation, metric_ablation, lgbm_alpha_ablation, lgbm_alpha_confirmation, tune, p0_experiments, validate, fit, all")
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
        force=True,
    )
    paths = Paths(root=Path(data_dir), artifacts=Path(output_dir))
    config = config or ModelConfig()
    validate_model_target_config(config)
    validate_early_stopping_metric(config)
    validate_lightgbm_huber_alpha(config)
    train, test, sample, coordinates, environment = load_inputs(paths)
    audit = audit_inputs(train, test, sample, coordinates, environment)
    LOGGER.info("Input audit passed:\n%s", json.dumps(audit, indent=2))
    result = {"mode": mode, "audit": audit}
    if mode == "audit":
        return result

    if mode in {"ablation", "target_ablation", "metric_ablation", "lgbm_alpha_ablation", "lgbm_alpha_confirmation", "tune", "p0_experiments"}:
        # Cheap diagnostic modes: build features once, run a lightweight study, and
        # write results next to (not inside) the heavy handoff run directories --
        # no point zipping up a multi-GB handoff for a sanity-check run.
        train_features, test_features, upstream_weights = build_feature_tables(
            train, test, coordinates, environment, paths, config
        )
        columns = feature_columns(train_features)
        stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        light_dir = paths.artifacts / f"{mode}_{stamp}"
        light_dir.mkdir(parents=True, exist_ok=True)
        if mode == "lgbm_alpha_confirmation":
            table = run_lgbm_alpha_confirmation(
                train_features, columns, config, light_dir
            )
            result["lgbm_alpha_confirmation_results"] = table
            result["lgbm_alpha_confirmation_dir"] = str(light_dir)
        elif mode == "lgbm_alpha_ablation":
            table = run_lgbm_alpha_ablation(
                train_features, columns, config, light_dir
            )
            result["lgbm_alpha_ablation_results"] = table
            result["lgbm_alpha_ablation_dir"] = str(light_dir)
        elif mode == "metric_ablation":
            table = run_metric_ablation(
                train_features, columns, config, light_dir
            )
            result["metric_ablation_results"] = table
            result["metric_ablation_dir"] = str(light_dir)
        elif mode == "target_ablation":
            table = run_target_mode_ablation(
                train_features, columns, config, light_dir
            )
            result["target_ablation_results"] = table
            result["target_ablation_dir"] = str(light_dir)
        elif mode == "ablation":
            table = run_ablation_study(
                train_features, columns, config, light_dir,
                feature_importance_source=config.feature_importance_source,
            )
            result["ablation_results"] = table
            result["ablation_dir"] = str(light_dir)
        elif mode == "tune":
            table = run_hyperparam_sweep(train_features, columns, config)
            table.to_csv(light_dir / "hyperparam_sweep.csv", index=False)
            result["sweep_results"] = table
            result["sweep_dir"] = str(light_dir)
        else:
            if config.prune_features:
                columns, prune_report = apply_feature_pruning(
                    columns, config.feature_importance_source, config.prune_keep_fraction
                )
                write_json(light_dir / "feature_pruning_report.json", prune_report)
            p0_result = run_p0_experiments(
                train_features, test_features, columns, config, light_dir
            )
            result.update(p0_result)
            result["p0_experiment_dir"] = str(light_dir)
        return result

    run_dir = create_run_directory(paths.artifacts, mode)
    attach_run_logger(run_dir)
    result["run_id"] = run_dir.name
    result["run_dir"] = str(run_dir)
    manifest = {
        "schema_version": HANDOFF_SCHEMA_VERSION,
        "run_id": run_dir.name,
        "mode": mode,
        "status": "in_progress",
        "stage": "initialized",
        "started_at_utc": utc_now_iso(),
        "input_dir": str(paths.root),
        "output_root": str(paths.artifacts),
        "ensemble": [m.name for m in config.ensemble],
        "use_spike_weights": config.use_spike_weights,
        "prune_features": config.prune_features,
        "core_handoff_files": [
            "manifest.json", "config.json", "runtime_versions.json",
            "input_inventory.json", "data/train_features.*",
            "data/test_features.*", "data/train_sample_diagnostics.*",
            "metrics/environment_drift.csv", "run.log",
        ],
        "mode_dependent_files": [
            "metrics/oof_predictions.*",
            "metrics/validation_metrics.csv", "metrics/member_metrics.csv",
            "metrics/fold_station_metrics.csv", "metrics/residual_slices.csv",
            "metrics/feature_importance.csv", "metrics/final_feature_importance.csv",
            "training_progress/", "models/", "submission.csv", "submission_quantiles.csv",
        ],
    }
    write_json(run_dir / "manifest.json", manifest)
    write_json(run_dir / "data_audit.json", audit)
    write_json(run_dir / "runtime_versions.json", runtime_versions())
    write_json(run_dir / "input_inventory.json", {"files": input_inventory(paths)})
    write_json(run_dir / "config.json", {
        "ensemble": [asdict(m) for m in config.ensemble],
        "folds": FOLDS,
        "mode": mode,
        "huber_delta": config.huber_delta,
        "lightgbm_huber_alpha": config.lightgbm_huber_alpha,
        "target_mode": config.target_mode,
        "early_stopping_rounds": config.early_stopping_rounds,
        "early_stopping_metric": config.early_stopping_metric,
        "early_stopping_metric": config.early_stopping_metric,
        "early_stopping_metric": config.early_stopping_metric,
        "use_spike_weights": config.use_spike_weights,
        "prune_features": config.prune_features,
        "feature_importance_source": config.feature_importance_source,
        "prune_keep_fraction": config.prune_keep_fraction,
        "enable_quantile_model": config.enable_quantile_model,
        "quantile_levels": config.quantile_levels,
        "enable_harmonic_feature": config.enable_harmonic_feature,
        "enable_anchor_feature": config.enable_anchor_feature,
        "frozen_anchor_off_feature_count": (
            len(ANCHOR_OFF_FROZEN_FEATURE_COLUMNS)
            if not config.enable_anchor_feature else None
        ),
        "enable_learned_ensemble_weights": config.enable_learned_ensemble_weights,
        "blend_ridge_strength": config.blend_ridge_strength,
        "blend_fit_folds": config.blend_fit_folds,
        "requested_iterations_by_member": iterations_by_member,
        "save_feature_matrices": save_feature_matrices,
    })
    write_handoff_readme(run_dir)

    try:
        train_features, test_features, upstream_weights = build_feature_tables(
            train, test, coordinates, environment, paths, config
        )
        columns = feature_columns(train_features)
        if config.prune_features:
            columns, prune_report = apply_feature_pruning(
                columns, config.feature_importance_source, config.prune_keep_fraction
            )
            write_json(run_dir / "feature_pruning_report.json", prune_report)
        columns = enforce_frozen_anchor_off_columns(
            train_features, columns, config
        )
        if not config.enable_anchor_feature:
            write_json(run_dir / "anchor_off_frozen_feature_report.json", {
                "applied": True,
                "feature_count": len(columns),
                "removed_anchor_features": list(ANCHOR_FEATURE_COLUMNS),
                "features": columns,
            })
        save_pretraining_handoff(
            run_dir, train_features, test_features, columns, upstream_weights, config,
            save_feature_matrices=save_feature_matrices,
        )
        LOGGER.info("Built %d leakage-safe features", len(columns))
        result["feature_count"] = len(columns)
        recommended = None
        current_ensemble_weights = None
        if mode in {"validate", "all"}:
            update_manifest(
                run_dir, stage="validation_training_started",
                training_started_at_utc=utc_now_iso(),
            )
            report, recommended = run_validation(
                train_features, columns, config, run_dir
            )
            result["validation"] = report
            result["recommended_iterations"] = recommended
            weight_path = run_dir / "metrics" / "ensemble_weights.json"
            if weight_path.exists():
                weight_payload = json.loads(weight_path.read_text(encoding="utf-8"))
                current_ensemble_weights = weight_payload.get("final_weights")
                result["ensemble_weights"] = current_ensemble_weights
            LOGGER.info("Validation report:\n%s", report.to_string(index=False))
        if mode in {"fit", "all"}:
            prior_recommended = find_latest_recommended_iterations(paths.artifacts)
            if iterations_by_member is not None:
                final_iterations, iteration_source = iterations_by_member, "explicit"
            elif recommended is not None:
                final_iterations, iteration_source = recommended, "current_validation"
            elif prior_recommended is not None:
                final_iterations, iteration_source = prior_recommended, "latest_prior_validation"
            else:
                final_iterations = {m.name: config.default_iterations for m in config.ensemble}
                iteration_source = "config_default"
            update_manifest(
                run_dir, stage="final_training_started",
                final_training_started_at_utc=utc_now_iso(),
                iteration_source=iteration_source,
            )
            if not config.enable_learned_ensemble_weights:
                # Hard guard: fit-only must not silently revive learned weights
                # from an older validation run when the champion config says OFF.
                final_ensemble_weights = None
                ensemble_weight_source = "equal_config_disabled"
            elif current_ensemble_weights is not None:
                final_ensemble_weights = current_ensemble_weights
                ensemble_weight_source = "current_validation"
            else:
                prior_weights = find_latest_ensemble_weights(paths.artifacts)
                final_ensemble_weights = prior_weights
                ensemble_weight_source = (
                    "latest_prior_validation" if prior_weights else "equal_fallback"
                )
            output = fit_final_and_submit(
                train_features, test_features, sample, columns, config,
                run_dir, final_iterations,
                ensemble_weights=final_ensemble_weights,
            )
            result["ensemble_weights"] = final_ensemble_weights
            result["ensemble_weight_source"] = ensemble_weight_source
            result["submission"] = str(output)
            result["final_iterations"] = final_iterations
            result["iteration_source"] = iteration_source
            LOGGER.info("Submission written to %s", output)
        result["handoff_zip"] = str(paths.artifacts / f"{run_dir.name}_complete.zip")
        result_summary = {
            key: (value.to_dict(orient="records") if isinstance(value, pd.DataFrame) else value)
            for key, value in result.items()
        }
        write_json(run_dir / "result_summary.json", result_summary)
        update_manifest(
            run_dir, status="complete", stage="complete",
            completed_at_utc=utc_now_iso(), handoff_zip=result["handoff_zip"],
        )
        handoff_zip = create_handoff_zip(run_dir, "complete")
        result["handoff_zip"] = str(handoff_zip)
        return result
    except Exception:
        error_text = traceback.format_exc()
        (run_dir / "error_traceback.txt").write_text(error_text, encoding="utf-8")
        LOGGER.exception("Pipeline failed; preserving partial handoff")
        update_manifest(
            run_dir, status="failed", stage="failed",
            failed_at_utc=utc_now_iso(),
        )
        create_handoff_zip(run_dir, "failed")
        raise


## 11. Flow confirmation LightGBM alpha

Mode ini menguji `alpha=0.7` terhadap champion `alpha=1.5`, lalu menghitung blend OOF 50:50 tanpa melatih model tambahan.

Eksperimen otomatis menjalankan:

- Anggota: `lgbm_a` dan `lgbm_b`.
- Fold exact-season: `sep_2023` dan `sep_2024`.
- Safety fold: `jan_2025`.
- Alpha: `0.7` dan `1.5`.
- Blend: `50% prediksi alpha 0.7 + 50% prediksi alpha 1.5`.

Seluruh variant memakai target raw, early-stopping metric lama, frozen 84 fitur, `nino_34` ON, harmonic ON, spike OFF, pruning OFF, anchor OFF, dan learned weighting OFF.

### Flow junior

1. Upload notebook terbaru dan pilih accelerator GPU seperti run sebelumnya.
2. Restart Session/Kernel.
3. Pastikan configuration cell menunjukkan `MODE="lgbm_alpha_confirmation"`.
4. Klik **Run All** satu kali. Jangan mengubah `LIGHTGBM_HUBER_ALPHA`; mode mengatur kedua alpha otomatis.
5. Ambil folder `lgbm_alpha_confirmation_<timestamp>`.
6. Serahkan seluruh folder kepada senior, khususnya:
   - `lgbm_alpha_confirmation_results.csv`
   - `lgbm_alpha_confirmation_oof.parquet`
   - `lgbm_alpha_confirmation_audit.json`
   - `lgbm_alpha_confirmation_decision.json`
7. Jangan menjalankan full validation, fit, atau upload submission meskipun JSON menyatakan lolos; keputusan akhir tetap diaudit senior.

Run ini melakukan 12 fit: 2 alpha × 2 LightGBM × 3 fold. Blend tidak menambah training.


In [ ]:
# Guard: Run All hanya melakukan audit dan TIDAK melatih model.
KAGGLE_DATA_DIR = Path("/kaggle/input/competitions/sebelas-maret-statistics-data-science-2026")
ON_KAGGLE = KAGGLE_DATA_DIR.exists()

MODE = "lgbm_alpha_confirmation"  # alpha 0.7 vs 1.5 + OOF blend
DATA_DIR = KAGGLE_DATA_DIR if ON_KAGGLE else Path("datasets")
OUTPUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path("artifacts")

# --- Toggle hasil review senior (lihat markdown tiap bagian untuk alasannya) ---
TARGET_MODE = "raw_robust"  # champion
EARLY_STOPPING_METRIC = "current"  # metric champion
LIGHTGBM_HUBER_ALPHA = 1.5  # baseline; mode alpha otomatis menguji seluruh grid
USE_SPIKE_WEIGHTS = False            # matikan untuk ablasi poin 3
PRUNE_FEATURES = False        # aktifkan setelah FEATURE_IMPORTANCE_SOURCE tersedia
FEATURE_IMPORTANCE_SOURCE = None  # pruning OFF; hindari path run lama
PRUNE_KEEP_FRACTION = 0.85
ENABLE_QUANTILE_MODEL = False       # CatBoost MultiQuantile tambahan (poin 4)
ENABLE_LEARNED_ENSEMBLE_WEIGHTS = False  # simplex OOF; tanpa target API
BLEND_RIDGE_STRENGTH = 0.05          # shrink ke equal weight saat member mirip
BLEND_FIT_FOLDS = ("sep_2023", "sep_2024")  # sama-sama horizon Sep-Mei
ENABLE_HARMONIC_FEATURE = True      # fitur baseline harmonik per pos (follow-up Wonogiri Dam)
HARMONIC_RIDGE_ALPHA = 5.0          # regularisasi Ridge; naikkan kalau kurva per pos terlalu "liar"
ENABLE_ANCHOR_FEATURE = False        # horizon_days + recent_anomaly + anchor_basis (follow-up station-aware stacking tim lain)
ANCHOR_RECENT_WINDOW_DAYS = 30.0    # jendela hari terakhir sebelum cutoff buat hitung recent_anomaly per pos
ANCHOR_TAU_DAYS = 365.0             # skala peluruhan horizon; makin besar makin lama anomali "dipercaya"
USE_GPU = True                      # otomatis fallback ke CPU per-member kalau GPU gagal/nggak ada
GPU_DEVICE_ID = "0:1"                 # index GPU Kaggle; ganti "0:1" untuk pakai 2 GPU sekaligus (T4 x2)
ITERATIONS_BY_MEMBER = None         # None = pakai rekomendasi validasi/prior run per member
SAVE_FEATURE_MATRICES = True        # disarankan untuk handoff penyempurnaan

CONFIG = ModelConfig(
    target_mode=TARGET_MODE,
    early_stopping_metric=EARLY_STOPPING_METRIC,
    lightgbm_huber_alpha=LIGHTGBM_HUBER_ALPHA,
    use_spike_weights=USE_SPIKE_WEIGHTS,
    prune_features=PRUNE_FEATURES,
    feature_importance_source=str(FEATURE_IMPORTANCE_SOURCE) if FEATURE_IMPORTANCE_SOURCE else None,
    prune_keep_fraction=PRUNE_KEEP_FRACTION,
    enable_quantile_model=ENABLE_QUANTILE_MODEL,
    enable_learned_ensemble_weights=ENABLE_LEARNED_ENSEMBLE_WEIGHTS,
    blend_ridge_strength=BLEND_RIDGE_STRENGTH,
    blend_fit_folds=BLEND_FIT_FOLDS,
    enable_harmonic_feature=ENABLE_HARMONIC_FEATURE,
    harmonic_ridge_alpha=HARMONIC_RIDGE_ALPHA,
    enable_anchor_feature=ENABLE_ANCHOR_FEATURE,
    anchor_recent_window_days=ANCHOR_RECENT_WINDOW_DAYS,
    anchor_tau_days=ANCHOR_TAU_DAYS,
    use_gpu=USE_GPU,
    gpu_device_id=GPU_DEVICE_ID,
)

print(f"Runtime: {'Kaggle' if ON_KAGGLE else 'local'}")
print(f"Input : {DATA_DIR}")
print(f"Output: {OUTPUT_DIR}")
print(f"Ensemble members ({len(CONFIG.ensemble)}): {[m.name for m in CONFIG.ensemble]}")
print(f"target_mode={CONFIG.target_mode} early_stopping_metric={CONFIG.early_stopping_metric} lightgbm_huber_alpha={CONFIG.lightgbm_huber_alpha} use_spike_weights={CONFIG.use_spike_weights} prune_features={CONFIG.prune_features} "
      f"enable_quantile_model={CONFIG.enable_quantile_model} enable_harmonic_feature={CONFIG.enable_harmonic_feature} "
      f"enable_anchor_feature={CONFIG.enable_anchor_feature} use_gpu={CONFIG.use_gpu}")


Runtime: Kaggle
Input : /kaggle/input/competitions/sebelas-maret-statistics-data-science-2026
Output: /kaggle/working
Ensemble members (5): ['cb_shallow', 'cb_base', 'cb_deep', 'lgbm_a', 'lgbm_b']
use_spike_weights=False prune_features=True enable_quantile_model=False enable_harmonic_feature=True enable_anchor_feature=True use_gpu=True


## 12. Jalankan mode yang dipilih


In [70]:
result = run_pipeline(
    mode=MODE,
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    config=CONFIG,
    iterations_by_member=ITERATIONS_BY_MEMBER,
    save_feature_matrices=SAVE_FEATURE_MATRICES,
)
result


2026-07-20 06:36:05,662 | INFO | Input audit passed:
{
  "train_rows": 84396,
  "test_rows": 21780,
  "environment_rows": 888480,
  "stations": 30,
  "train_start": "2023-01-01 06:00:00",
  "train_end": "2025-09-18 18:00:00",
  "test_start": "2025-09-19 06:00:00",
  "test_end": "2026-05-18 18:00:00",
  "environment_missing_cells": 20880,
  "solar_minus_999_cells": 100080
}
2026-07-20 06:36:12,420 | INFO | Feature pruning: kept 87/102 features (dropped 15)
2026-07-20 06:36:13,802 | INFO | Built 87 leakage-safe features
Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.0592706	test: 1.7188717	best: 1.7188717 (0)	total: 38.6ms	remaining: 1m 9s
100:	learn: 0.6352224	test: 1.4991557	best: 1.4991557 (100)	total: 2.67s	remaining: 45s
200:	learn: 0.5945423	test: 1.4899801	best: 1.4899801 (200)	total: 5.19s	remaining: 41.3s
300:	learn: 0.5705905	test: 1.4869188	best: 1.4855726 (257)	total: 7.68s	remaining: 38.2s
400:	learn: 0.5543127	test: 1.4840206	best: 1.4828934 (369)	total: 10.2s	remaining: 35.5s
500:	learn: 0.5413845	test: 1.4797769	best: 1.4788844 (473)	total: 12.7s	remaining: 32.8s
600:	learn: 0.5320156	test: 1.4806009	best: 1.4766645 (584)	total: 15.1s	remaining: 30.1s
700:	learn: 0.5230579	test: 1.4790931	best: 1.4766645 (584)	total: 17.7s	remaining: 27.7s
bestTest = 1.476664516
bestIteration = 584
Shrink model to first 585 iterations.


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.0767584	test: 1.7206880	best: 1.7206880 (0)	total: 54.4ms	remaining: 1m 37s
100:	learn: 0.6655400	test: 1.5534653	best: 1.5534653 (100)	total: 4.74s	remaining: 1m 19s
200:	learn: 0.6122148	test: 1.5138454	best: 1.5138454 (200)	total: 9.52s	remaining: 1m 15s
300:	learn: 0.5866077	test: 1.5035443	best: 1.5034390 (283)	total: 14.3s	remaining: 1m 11s
400:	learn: 0.5701655	test: 1.5020103	best: 1.5012823 (366)	total: 19s	remaining: 1m 6s
500:	learn: 0.5571401	test: 1.5009312	best: 1.5008462 (484)	total: 23.5s	remaining: 1m
600:	learn: 0.5459360	test: 1.4982705	best: 1.4981798 (599)	total: 28s	remaining: 55.9s
700:	learn: 0.5378396	test: 1.4976940	best: 1.4975646 (609)	total: 32.6s	remaining: 51.1s
800:	learn: 0.5300487	test: 1.4964115	best: 1.4964115 (800)	total: 37.1s	remaining: 46.3s
900:	learn: 0.5236337	test: 1.4947342	best: 1.4947296 (899)	total: 41.7s	remaining: 41.6s
1000:	learn: 0.5184888	test: 1.4935332	best: 1.4934275 (981)	total: 46.3s	remaining: 36.9s
1100:	learn: 0.

Default metric period is 5 because MAE is/are not implemented for GPU


...

0:	learn: 0.9128109	total: 24.2ms	remaining: 33.5s
100:	learn: 0.5376379	total: 2.15s	remaining: 27.3s
200:	learn: 0.4893277	total: 4.4s	remaining: 25.9s
300:	learn: 0.4627764	total: 6.58s	remaining: 23.7s
400:	learn: 0.4450943	total: 8.79s	remaining: 21.5s
500:	learn: 0.4315933	total: 11s	remaining: 19.3s
600:	learn: 0.4192524	total: 13.2s	remaining: 17.1s
700:	learn: 0.4087014	total: 15.4s	remaining: 15s
800:	learn: 0.3995492	total: 17.7s	remaining: 12.8s
900:	learn: 0.3913891	total: 19.9s	remaining: 10.6s
1000:	learn: 0.3843715	total: 22s	remaining: 8.4s
1100:	learn: 0.3773129	total: 24.3s	remaining: 6.22s
1200:	learn: 0.3716222	total: 26.5s	remaining: 4.01s
1300:	learn: 0.3663740	total: 28.7s	remaining: 1.81s
1382:	learn: 0.3623408	total: 30.6s	remaining: 0us


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 0.8967386	total: 37.4ms	remaining: 9.69s
100:	learn: 0.4764976	total: 2.42s	remaining: 3.81s
200:	learn: 0.4350763	total: 4.63s	remaining: 1.36s
259:	learn: 0.4165597	total: 5.94s	remaining: 0us


2026-07-20 06:44:54,347 | INFO | Submission written to /kaggle/working/handoff_20260720T063605Z_all/submission.csv


{'mode': 'all',
 'audit': {'train_rows': 84396,
  'test_rows': 21780,
  'environment_rows': 888480,
  'stations': 30,
  'train_start': '2023-01-01 06:00:00',
  'train_end': '2025-09-18 18:00:00',
  'test_start': '2025-09-19 06:00:00',
  'test_end': '2026-05-18 18:00:00',
  'environment_missing_cells': 20880,
  'solar_minus_999_cells': 100080},
 'run_id': 'handoff_20260720T063605Z_all',
 'run_dir': '/kaggle/working/handoff_20260720T063605Z_all',
 'feature_count': 87,
 'validation':        fold  train_rows  valid_rows  valid_stations  \
 0  sep_2023       22336       20696              29   
 1  may_2024       43032       21313              29   
 2  sep_2024       53838       19514              30   
 3  jan_2025       64817       19579              30   
 
    isolated_spikes_downweighted  use_spike_weights  \
 0                            22              False   
 1                            30              False   
 2                            29              False   
 3           